In [198]:
# --------------------------------------------------------------------------- #
# Step 4.5: Repurchase-cycle estimation from real transaction data
# --------------------------------------------------------------------------- #
#
# Goal: derive an evidence-based post_period_weeks per commodity, instead of
# the fixed 4-week placeholder in Config. Per document 3's rule, the horizon
# must be at least as long as the product's natural purchase frequency, and
# per the PDF's baseline rule, this must be estimated ONLY on ordinary,
# unpromoted periods -- so we exclude any day that falls inside ANY campaign's
# window (not just the household's own linked campaigns), since a household
# can be exposed to promotional pricing/marketing spillover even from
# campaigns it isn't formally linked to.
def estimate_repurchase_cycles(cfg, campaign_desc, product, min_purchases=3):
    """
    Streams transaction_data.csv once. For each household x commodity pair,
    computes inter-purchase day gaps using ONLY unpromoted days -- i.e. days
    outside every campaign's [START_DAY, END_DAY] window.

    IMPORTANT FIX: earlier version restricted the whole search to the
    224-719 campaign-season window and THEN excluded promoted days within
    it -- since campaigns overlap heavily, that window can be almost fully
    covered by promotions, leaving zero unpromoted days. This version uses
    the FULL transaction history: days outside [day_min, day_max] are
    unpromoted by definition and need no lookup; only days inside that
    range go through the per-day promoted check.
    """
    log.info("Estimating repurchase cycles from unpromoted transaction days")

    windows = campaign_desc[["START_DAY", "END_DAY"]].to_numpy()
    day_min, day_max = int(campaign_desc["START_DAY"].min()), int(campaign_desc["END_DAY"].max())

    def _is_promoted_day(day):
        return np.any((windows[:, 0] <= day) & (day <= windows[:, 1]))

    all_days = np.arange(day_min, day_max + 1)
    promoted_lookup = pd.Series(
        [_is_promoted_day(d) for d in all_days], index=all_days
    )

    product_commodity = product.set_index("PRODUCT_ID")["COMMODITY_DESC"]
    purchase_days = {}
    total_rows_seen = 0
    total_unpromoted_rows = 0

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        total_rows_seen += len(chunk)
        chunk["COMMODITY_DESC"] = chunk["PRODUCT_ID"].map(product_commodity)
        chunk = chunk.dropna(subset=["COMMODITY_DESC"])

        # Days inside the campaign season: check the lookup.
        # Days outside it: unpromoted by definition, keep automatically.
        inside_season = chunk["DAY"].between(day_min, day_max)
        is_promoted = pd.Series(False, index=chunk.index)
        if inside_season.any():
            is_promoted.loc[inside_season] = promoted_lookup.reindex(
                chunk.loc[inside_season, "DAY"]
            ).to_numpy()

        chunk = chunk[~is_promoted]
        total_unpromoted_rows += len(chunk)
        if chunk.empty:
            continue

        grp = chunk.groupby(["household_key", "COMMODITY_DESC"])["DAY"].apply(
            lambda s: sorted(s.unique())
        )
        for (hh, commodity), days in grp.items():
            key = (hh, commodity)
            purchase_days.setdefault(key, [])
            purchase_days[key].extend(days)

    log.info(f"Unpromoted rows kept: {total_unpromoted_rows} / {total_rows_seen} "
              f"({total_unpromoted_rows/total_rows_seen:.1%})")

    rows = []
    for (hh, commodity), days in purchase_days.items():
        days = sorted(set(days))
        if len(days) < min_purchases:
            continue
        gaps = np.diff(days)
        rows.append({"COMMODITY_DESC": commodity, "HOUSEHOLD_KEY": hh, "MEDIAN_GAP_DAYS": np.median(gaps)})

    gap_df = pd.DataFrame(rows)
    if gap_df.empty:
        log.warning("No household-commodity pairs met min_purchases threshold on unpromoted "
                     "days -- returning empty summary. Check total_unpromoted_rows above.")
        return pd.DataFrame(columns=["COMMODITY_DESC", "n_households", "median_gap", "p75_gap", "p90_gap"])

    summary = (
        gap_df.groupby("COMMODITY_DESC")["MEDIAN_GAP_DAYS"]
        .agg(n_households="count", median_gap="median",
             p75_gap=lambda s: np.percentile(s, 75),
             p90_gap=lambda s: np.percentile(s, 90))
        .reset_index()
        .sort_values("n_households", ascending=False)
    )
    log.info(f"Repurchase-cycle estimates computed for {len(summary)} commodities "
             f"(households with >= {min_purchases} unpromoted purchases only)")
    return summary
cfg = Config(Path("data"))
tables = load_reference_tables(cfg)
repurchase_cycles = estimate_repurchase_cycles(
    cfg, tables["campaign_desc"], tables["product"], min_purchases=3
)
repurchase_cycles.to_csv("pipeline_output/repurchase_cycles_by_commodity.csv", index=False)
repurchase_cycles.head(20)

2026-08-07 10:10:56,553 | INFO | Loading reference tables
2026-08-07 10:10:56,636 | INFO | Estimating repurchase cycles from unpromoted transaction days
2026-08-07 10:11:00,041 | INFO | Unpromoted rows kept: 627727 / 2595732 (24.2%)
2026-08-07 10:11:00,522 | INFO | Repurchase-cycle estimates computed for 270 commodities (households with >= 3 unpromoted purchases only)


,COMMODITY_DESC,n_households,median_gap,p75_gap,p90_gap
112,FLUID MILK PRODUCTS,1565,13.5,24.00,42.0
14,BAKED BREAD/BUNS/ROLLS,1502,14.0,24.00,39.5
239,SOFT DRINKS,1488,12.0,22.50,38.0
48,CHEESE,1284,16.5,29.00,46.5
13,BAG SNACKS,1181,16.5,28.00,46.0
23,BEEF,1082,16.0,28.00,46.5
93,EGGS,855,22.0,36.00,53.0
256,TROPICAL FRUIT,853,16.0,28.00,45.0
61,COLD CEREAL,799,20.0,32.00,50.0
213,REFRGRATD JUICES/DRNKS,777,17.0,30.00,50.5


In [199]:
# --------------------------------------------------------------------------- #
# Step 4.6b: Discount-depth proxy + storability heuristic + campaign-length
# adjustment, combined into the post-window calculation.
# --------------------------------------------------------------------------- #
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd

def compute_discount_depth_proxy(cfg, campaign_desc: pd.DataFrame, coupon: pd.DataFrame) -> pd.DataFrame:
    """
    Computes a discount depth proxy at the campaign level using coupon allocation data.
    
    Args:
        cfg: Configuration object or dataclass containing pipeline parameters.
        campaign_desc: DataFrame containing campaign metadata (e.g., CAMPAIGN, START_DAY).
        coupon: Deduplicated DataFrame mapping coupons to campaigns and products.
        
    Returns:
        DataFrame containing original campaign metadata appended with DISCOUNT_DEPTH_PROXY.
    """
    
    # 1. Verify the necessary keys exist to prevent KeyErrors
    if 'CAMPAIGN' not in coupon.columns:
        raise KeyError("The coupon table must contain a 'CAMPAIGN' column to aggregate.")
        
    # 2. Compute the proxy metric 
    # Default assumption: The depth proxy is the number of unique products on promotion
    if 'PRODUCT_ID' in coupon.columns:
        proxy_agg = coupon.groupby('CAMPAIGN', as_index=False).agg(
            DISCOUNT_DEPTH_PROXY=('PRODUCT_ID', 'nunique')
        )
    elif 'COUPON_UPC' in coupon.columns:
        # Fallback: Count unique coupons if PRODUCT_ID isn't available
        proxy_agg = coupon.groupby('CAMPAIGN', as_index=False).agg(
            DISCOUNT_DEPTH_PROXY=('COUPON_UPC', 'nunique')
        )
    else:
        # Generic fallback: Simply count the number of rows (allocations) per campaign
        proxy_agg = coupon.groupby('CAMPAIGN', as_index=False).size()
        proxy_agg.rename(columns={'size': 'DISCOUNT_DEPTH_PROXY'}, inplace=True)
        
    # 3. Merge the computed proxy back into the campaign metadata table
    campaign_metrics = campaign_desc.merge(
        proxy_agg, 
        on='CAMPAIGN', 
        how='left'
    )
    
    # 4. Fill NaNs with 0 for campaigns that had no associated coupons
    campaign_metrics['DISCOUNT_DEPTH_PROXY'] = campaign_metrics['DISCOUNT_DEPTH_PROXY'].fillna(0)
    
    return campaign_metrics

def flag_storability(product, perishable_keywords=None):
    """
    Heuristic ONLY -- no perishability field exists in product.csv. Flags a
    commodity as perishable based on department/commodity text matching.
    This is an assumption to be validated, not a fact derived from the data.
    """
    if perishable_keywords is None:
        perishable_keywords = ["PRODUCE", "MEAT", "DAIRY", "DELI", "SEAFOOD", "BAKERY", "FLORAL"]

    df = product[["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC"]].copy()
    text = (df["DEPARTMENT"].fillna("") + " " + df["COMMODITY_DESC"].fillna("")).str.upper()
    df["IS_PERISHABLE"] = text.apply(lambda t: any(kw in t for kw in perishable_keywords))
    commodity_perishable = df.groupby("COMMODITY_DESC")["IS_PERISHABLE"].agg(lambda s: s.mean() > 0.5)
    log.info(f"Storability heuristic: {commodity_perishable.sum()}/{len(commodity_perishable)} "
             f"commodities flagged perishable (keyword match on DEPARTMENT/COMMODITY_DESC)")
    return commodity_perishable


def build_campaign_post_windows_v3(
    repurchase_cycles, coupon, product, campaign_desc, discount_proxy,
    default_weeks=4, cap_weeks=16,
    storability_discount_factor=0.25,   # dialed down from 0.5 -- was overriding real evidence
    discount_depth_multiplier=4,
    length_floor_fraction=0.5,          # NEW: floor is a FRACTION of campaign duration, not the full length
):
    """
    v3 fix: LENGTH_FLOOR is now `length_floor_fraction * campaign_duration_weeks`,
    not the full duration. Rationale: the post-window measures time AFTER the
    campaign ends -- there's no principled reason it must match the campaign's
    own length 1:1. A long campaign already gets its during-window measured at
    full length regardless of this parameter; this floor only exists so a very
    long campaign doesn't get an implausibly short post-window from a low
    repurchase-gap commodity mix.
    """
    product_commodity = product.set_index("PRODUCT_ID")["COMMODITY_DESC"]
    elig = coupon[["CAMPAIGN", "PRODUCT_ID"]].drop_duplicates()
    elig["COMMODITY_DESC"] = elig["PRODUCT_ID"].map(product_commodity)
    gap_lookup = repurchase_cycles.set_index("COMMODITY_DESC")["p75_gap"]
    commodity_perishable = flag_storability(product)
    duration_weeks = (campaign_desc.set_index("CAMPAIGN")["END_DAY"]
                       - campaign_desc.set_index("CAMPAIGN")["START_DAY"] + 1) / 7.0

    breakdown = []
    for campaign, group in elig.groupby("CAMPAIGN"):
        commodities = set(group["COMMODITY_DESC"].dropna())
        gaps = gap_lookup.reindex(list(commodities)).dropna()
        base_weeks = np.ceil(gaps.max() / 7.0) if not gaps.empty else default_weeks

        length_floor = np.ceil(duration_weeks.get(campaign, 0) * length_floor_fraction)
        after_length = max(base_weeks, length_floor)

        d = discount_proxy.get(campaign, np.nan)
        depth_add = discount_depth_multiplier * d if pd.notna(d) else 0.0
        after_depth = after_length + depth_add

        perishable_share = commodity_perishable.reindex(list(commodities)).fillna(False).mean() if commodities else 0.0
        shrink_factor = 1 - storability_discount_factor * perishable_share
        final_weeks = int(np.ceil(min(after_depth * shrink_factor, cap_weeks)))
        final_weeks = max(final_weeks, 2)

        breakdown.append({
            "CAMPAIGN": campaign, "BASE_WEEKS": base_weeks, "LENGTH_FLOOR": length_floor,
            "DISCOUNT_PROXY": d, "DEPTH_ADD": round(depth_add, 1),
            "PERISHABLE_SHARE": round(perishable_share, 2), "FINAL_WEEKS": final_weeks,
            "HIT_CAP": after_depth * shrink_factor > cap_weeks,   # NEW: flag when the cap is actually binding
        })

    breakdown_df = pd.DataFrame(breakdown).sort_values("CAMPAIGN")
    n_capped = breakdown_df["HIT_CAP"].sum()
    if n_capped:
        log.warning(f"{n_capped} campaign(s) hit cap_weeks={cap_weeks} -- inspect these individually, "
                     f"the cap is discarding evidence rather than reflecting it: "
                     f"{breakdown_df.loc[breakdown_df['HIT_CAP'], 'CAMPAIGN'].tolist()}")
    log.info(f"Post-window range: {breakdown_df['FINAL_WEEKS'].min()}-{breakdown_df['FINAL_WEEKS'].max()} weeks")
    return breakdown_df.set_index("CAMPAIGN")["FINAL_WEEKS"].to_dict(), breakdown_df
discount_proxy_df = compute_discount_depth_proxy(cfg, tables["campaign_desc"], dedupe_coupon_table(tables["coupon"]))

raw = discount_proxy_df.set_index("CAMPAIGN")["DISCOUNT_DEPTH_PROXY"]
discount_proxy = ((raw - raw.min()) / (raw.max() - raw.min())).to_dict()

campaign_post_windows_v3, post_window_breakdown_v3 = build_campaign_post_windows_v3(
    repurchase_cycles, dedupe_coupon_table(tables["coupon"]), tables["product"],
    tables["campaign_desc"], discount_proxy,
    discount_depth_multiplier=4,   # now interpreted as "up to 4 extra weeks" at proxy=1.0
)
post_window_breakdown_v3

2026-08-07 10:11:00,566 | INFO | coupon.csv: dropped 5164 exact-duplicate rows (4.15%)
2026-08-07 10:11:00,576 | INFO | coupon.csv: dropped 5164 exact-duplicate rows (4.15%)
2026-08-07 10:11:00,674 | INFO | Storability heuristic: 86/308 commodities flagged perishable (keyword match on DEPARTMENT/COMMODITY_DESC)
2026-08-07 10:11:00,706 | INFO | Post-window range: 5-13 weeks


,CAMPAIGN,BASE_WEEKS,LENGTH_FLOOR,DISCOUNT_PROXY,DEPTH_ADD,PERISHABLE_SHARE,FINAL_WEEKS,HIT_CAP
0,1,6.0,3.0,0.010253,0.0,0.10,6,False
1,2,7.0,3.0,0.007950,0.0,0.27,7,False
2,3,8.0,5.0,0.013709,0.1,0.07,8,False
3,4,7.0,3.0,0.005113,0.0,0.00,8,False
4,5,10.0,3.0,0.011939,0.0,0.00,11,False
5,6,5.0,3.0,0.000000,0.0,0.00,5,False
6,7,8.0,3.0,0.006770,0.0,0.08,8,False
7,8,10.0,4.0,0.482050,1.9,0.39,11,False
8,9,8.0,3.0,0.022670,0.1,0.10,8,False
9,10,8.0,3.0,0.010534,0.0,0.00,9,False


In [200]:
def build_universal_outcomes(
    cfg,
    tables,
    post_windows_by_campaign: dict | None = None,   # {CAMPAIGN: weeks}, e.g. campaign_post_windows_v3
    pre_period_weeks: int = 4,
    default_post_weeks: int = 4,
) -> pd.DataFrame:
    """
    Aggregates household-level transactions across pre / during / post windows
    to construct the full outcome vector, including Y_PRE_SALES (baseline for
    DiD) and Y_CATEGORY_SALES (needed to compute true same-commodity Y_RIVAL_SALES).

    post_windows_by_campaign lets each campaign use its own evidence-based
    post-window length (from build_campaign_post_windows_v3) instead of one
    flat value for every campaign -- this also fixes item 1 from the review,
    since campaign_post_windows_v3 was computed but never actually wired in.
    """
    log.info("Constructing universal outcome matrix across household-campaign windows...")

    campaign_desc = tables["campaign_desc"]
    coupon = tables["coupon"]
    product = tables["product"]

    coupon_dedup = coupon.drop_duplicates(subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"])
    product_commodity = product.set_index("PRODUCT_ID")["COMMODITY_DESC"]

    elig = coupon_dedup[["CAMPAIGN", "PRODUCT_ID"]].drop_duplicates()
    elig["COMMODITY_DESC"] = elig["PRODUCT_ID"].map(product_commodity)
    elig_map = elig.groupby("CAMPAIGN")["PRODUCT_ID"].apply(set).to_dict()
    elig_commodity_map = elig.groupby("CAMPAIGN")["COMMODITY_DESC"].apply(lambda s: set(s.dropna())).to_dict()

    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]].to_dict("index")

    if post_windows_by_campaign is None:
        post_windows_by_campaign = {}

    outcomes = {}
    METRIC_KEYS = [
        "Y_ELIGIBLE_SALES", "Y_ELIGIBLE_UNITS", "Y_CATEGORY_SALES",
        "Y_RIVAL_SALES", "Y_PRE_SALES", "Y_POST_SALES",
    ]

    def _init(key):
        if key not in outcomes:
            outcomes[key] = {k: 0.0 for k in METRIC_KEYS}
        return outcomes[key]

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk = chunk[chunk["QUANTITY"] > 0].copy()
        chunk["COMMODITY_DESC"] = chunk["PRODUCT_ID"].map(product_commodity)

        for camp_id, bounds in windows.items():
            start, end = bounds["START_DAY"], bounds["END_DAY"]
            pre_start = start - pre_period_weeks * 7
            post_weeks = post_windows_by_campaign.get(camp_id, default_post_weeks)
            post_end = end + int(post_weeks) * 7

            sub = chunk[(chunk["DAY"] >= pre_start) & (chunk["DAY"] <= post_end)]
            if sub.empty:
                continue

            elig_skus = elig_map.get(camp_id, set())
            elig_commodities = elig_commodity_map.get(camp_id, set())

            # --- Pre-period baseline (NEW -- this was missing entirely) ---
            pre = sub[(sub["DAY"] >= pre_start) & (sub["DAY"] < start)]
            if not pre.empty:
                for hh, val in pre.groupby("household_key")["SALES_VALUE"].sum().items():
                    _init((hh, camp_id))["Y_PRE_SALES"] += val

            # --- Campaign window (during) ---
            during = sub[(sub["DAY"] >= start) & (sub["DAY"] <= end)]
            if not during.empty:
                is_elig = during["PRODUCT_ID"].isin(elig_skus)
                same_commodity = during["COMMODITY_DESC"].isin(elig_commodities)

                elig_df = during[is_elig]
                for hh, g in elig_df.groupby("household_key"):
                    m = _init((hh, camp_id))
                    m["Y_ELIGIBLE_SALES"] += g["SALES_VALUE"].sum()
                    m["Y_ELIGIBLE_UNITS"] += g["QUANTITY"].sum()

                # NEW: category sales = same commodity as eligible products,
                # regardless of eligible/non-eligible split
                category_df = during[same_commodity]
                for hh, g in category_df.groupby("household_key"):
                    _init((hh, camp_id))["Y_CATEGORY_SALES"] += g["SALES_VALUE"].sum()

                # FIXED: rival = same commodity, non-eligible only -- not the
                # whole non-eligible basket. This matches cell 0's definition
                # and doc 3's "competing SKUs" (Section 4), not "everything else
                # the household bought that week."
                rival_df = during[same_commodity & ~is_elig]
                for hh, g in rival_df.groupby("household_key"):
                    _init((hh, camp_id))["Y_RIVAL_SALES"] += g["SALES_VALUE"].sum()

            # --- Post-campaign window ---
            post = sub[(sub["DAY"] > end) & (sub["DAY"] <= post_end)]
            if not post.empty:
                for hh, val in post.groupby("household_key")["SALES_VALUE"].sum().items():
                    _init((hh, camp_id))["Y_POST_SALES"] += val

    records = [
        {"household_key": hh, "CAMPAIGN": camp_id, **metrics}
        for (hh, camp_id), metrics in outcomes.items()
    ]
    df_outcomes = pd.DataFrame(records)
    log.info(f"Universal outcomes table created with {len(df_outcomes):,} household-campaign records.")
    return df_outcomes


# Build and export outcomes -- now passes campaign_post_windows_v3 (item 1 fix)
universal_outcomes = build_universal_outcomes(cfg, tables, post_windows_by_campaign=campaign_post_windows_v3)
universal_outcomes = build_universal_outcomes(
    cfg, tables, post_windows_by_campaign=campaign_post_windows_v3
)
universal_outcomes = universal_outcomes.rename(columns={"household_key": "HOUSEHOLD_KEY"})
universal_outcomes.to_csv("pipeline_output/universal_outcomes.csv", index=False)
universal_outcomes.head()


2026-08-07 10:11:00,769 | INFO | Constructing universal outcome matrix across household-campaign windows...
2026-08-07 10:11:05,944 | INFO | Universal outcomes table created with 69,566 household-campaign records.
2026-08-07 10:11:05,981 | INFO | Constructing universal outcome matrix across household-campaign windows...
2026-08-07 10:11:11,323 | INFO | Universal outcomes table created with 69,566 household-campaign records.


,HOUSEHOLD_KEY,CAMPAIGN,Y_ELIGIBLE_SALES,Y_ELIGIBLE_UNITS,Y_CATEGORY_SALES,Y_RIVAL_SALES,Y_PRE_SALES,Y_POST_SALES
0,72,1,8.47,3.0,80.25,71.78,430.01,566.79
1,83,1,4.78,2.0,4.78,0.00,73.16,24.05
2,97,1,1.67,1.0,21.17,19.50,231.19,233.56
3,287,1,0.00,0.0,34.71,34.71,296.60,162.88
4,290,1,0.00,0.0,30.73,30.73,505.61,432.65


In [201]:
"""
Campaign causal analysis pipeline.

Implements, in order:
  Part 1 : estimand definition (see project-breakdown-part1.md) + episode table
  Step 3 : transaction unit / zero audit
  Outcome construction (Y columns) from the transaction file
  Plan 1 : matched stacked event-study Difference-in-Differences

Assumptions baked in from prior analysis of the files (see project-breakdown-part1.md):
  - Treatment D=1 is CAMPAIGN ASSIGNMENT (campaign_table.csv), never redemption.
  - coupon.csv must be joined on (CAMPAIGN, COUPON_UPC), and deduplicated on
    (CAMPAIGN, COUPON_UPC, PRODUCT_ID) before counting eligible products.
  - Households linked to overlapping campaigns are flagged, not silently pooled.
  - Effects are reported per campaign-week, not raw campaign totals, because
    campaign duration ranges 33-162 days.

This has NOT been run against real data yet -- it was written against the
documented schemas only. Run scripts/smoke_test in this file's __main__
block after pointing CONFIG at your actual files, and inspect intermediate
outputs before trusting anything downstream.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("campaign_pipeline")


# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #

@dataclass
class Config:
    data_dir: Path
    campaign_desc_file: str = "campaign_desc.csv"
    campaign_table_file: str = "campaign_table.csv"
    coupon_file: str = "coupon.csv"
    coupon_redempt_file: str = "coupon_redempt.csv"
    product_file: str = "product.csv"
    hh_demographic_file: str = "hh_demographic.csv"
    transaction_file: str = "transaction_data.csv"

    # Analysis parameters -- these are choices, not facts from the files.
    # State them explicitly so they're easy to challenge/change.
    pre_period_weeks: int = 4          # weeks before campaign start used as baseline
    post_period_weeks: int = 4         # weeks after campaign end used for payback check
    transaction_chunksize: int = 1_000_000  # for the ~6M row file

    paths: dict = field(init=False)

    def __post_init__(self):
        self.paths = {
            "campaign_desc": self.data_dir / self.campaign_desc_file,
            "campaign_table": self.data_dir / self.campaign_table_file,
            "coupon": self.data_dir / self.coupon_file,
            "coupon_redempt": self.data_dir / self.coupon_redempt_file,
            "product": self.data_dir / self.product_file,
            "hh_demographic": self.data_dir / self.hh_demographic_file,
            "transaction": self.data_dir / self.transaction_file,
        }


# --------------------------------------------------------------------------- #
# 1. Loading
# --------------------------------------------------------------------------- #

def load_reference_tables(cfg: Config) -> dict[str, pd.DataFrame]:
    """Load the six small reference CSVs (not the transaction file)."""
    log.info("Loading reference tables")

    campaign_desc = pd.read_csv(cfg.paths["campaign_desc"])
    campaign_table = pd.read_csv(cfg.paths["campaign_table"])
    coupon = pd.read_csv(cfg.paths["coupon"])
    coupon_redempt = pd.read_csv(cfg.paths["coupon_redempt"])
    product = pd.read_csv(cfg.paths["product"])
    hh_demographic = pd.read_csv(cfg.paths["hh_demographic"])

    # Normalize column names defensively (source files have been observed to
    # vary in case/whitespace across this dataset family).
    for df in (campaign_desc, campaign_table, coupon, coupon_redempt, product, hh_demographic):
        df.columns = [c.strip().upper() for c in df.columns]

    return {
        "campaign_desc": campaign_desc,
        "campaign_table": campaign_table,
        "coupon": coupon,
        "coupon_redempt": coupon_redempt,
        "product": product,
        "hh_demographic": hh_demographic,
    }


def dedupe_coupon_table(coupon: pd.DataFrame) -> pd.DataFrame:
    """
    coupon.csv is an eligibility table with 5,164 exact-duplicate rows (4.15%)
    across (CAMPAIGN, COUPON_UPC, PRODUCT_ID) triplets. Drop exact duplicates
    only -- do NOT collapse legitimate one-coupon-to-many-product mappings.
    """
    before = len(coupon)
    coupon_dedup = coupon.drop_duplicates(
        subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"]
    ).copy()
    dropped = before - len(coupon_dedup)
    log.info(f"coupon.csv: dropped {dropped} exact-duplicate rows ({dropped/before:.2%})")
    return coupon_dedup


# --------------------------------------------------------------------------- #
# 2. Episode table (Part 1 deliverable)
# --------------------------------------------------------------------------- #

def build_episode_table(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Build the household x campaign episode table.
    D (treatment indicator) = row exists here = household assigned to campaign.
    Redemption is kept as a separate outcome-adjacent column, never as D.
    """
    log.info("Building episode table")

    campaign_table = tables["campaign_table"]
    campaign_desc = tables["campaign_desc"]
    coupon = dedupe_coupon_table(tables["coupon"])
    coupon_redempt = tables["coupon_redempt"]
    hh_demographic = tables["hh_demographic"]

    episodes = campaign_table.merge(
        campaign_desc[["CAMPAIGN", "DESCRIPTION", "START_DAY", "END_DAY"]].rename(
            columns={"DESCRIPTION": "CAMPAIGN_TYPE"}
        ),
        on="CAMPAIGN",
        how="left",
        validate="many_to_one",
    )
    episodes["DURATION_DAYS"] = episodes["END_DAY"] - episodes["START_DAY"] + 1

    # --- concurrent campaigns: count of *other* campaigns this household is
    # also linked to, whose windows overlap this campaign's window.
    episodes = _add_concurrent_campaign_count(episodes, campaign_desc)

    # --- eligible product count per campaign (post-dedup)
    eligible_counts = (
        coupon.groupby("CAMPAIGN")["PRODUCT_ID"].nunique().rename("ELIGIBLE_PRODUCT_COUNT")
    )
    episodes = episodes.merge(eligible_counts, on="CAMPAIGN", how="left")

    # --- redemption flag + first redemption day (household x campaign)
    redempt_agg = (
        coupon_redempt.groupby(["household_key".upper(), "CAMPAIGN"])["DAY"]
        .min()
        .rename("FIRST_REDEMPTION_DAY")
        .reset_index()
    )
    episodes = episodes.merge(redempt_agg, on=["HOUSEHOLD_KEY", "CAMPAIGN"], how="left")
    episodes["REDEEMED"] = episodes["FIRST_REDEMPTION_DAY"].notna().astype(int)

    # --- demographics (kept as explicit categories; missing stays missing)
    episodes = episodes.merge(hh_demographic, on="HOUSEHOLD_KEY", how="left")
    episodes["HAS_DEMOGRAPHICS"] = episodes["AGE_DESC"].notna().astype(int) if "AGE_DESC" in episodes else np.nan

    log.info(f"Episode table: {len(episodes)} rows, {episodes['HOUSEHOLD_KEY'].nunique()} households")
    return episodes


def _add_concurrent_campaign_count(episodes: pd.DataFrame, campaign_desc: pd.DataFrame) -> pd.DataFrame:
    """
    For every episode (household, campaign), count how many OTHER campaigns
    that same household is linked to whose [START_DAY, END_DAY] overlaps this
    campaign's window. O(n_households * campaigns_per_household^2) via groupby
    -- fine at 7,208 rows / max 17 campaigns per household.
    """
    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]]

    def _count_for_group(group: pd.DataFrame) -> pd.Series:
        camps = group["CAMPAIGN"].tolist()
        counts = []
        for c in camps:
            s0, e0 = windows.loc[c, "START_DAY"], windows.loc[c, "END_DAY"]
            n_overlap = 0
            for other in camps:
                if other == c:
                    continue
                s1, e1 = windows.loc[other, "START_DAY"], windows.loc[other, "END_DAY"]
                if s0 <= e1 and s1 <= e0:
                    n_overlap += 1
            counts.append(n_overlap)
        return pd.Series(counts, index=group.index)

    episodes = episodes.copy()
    episodes["N_CONCURRENT_CAMPAIGNS"] = (
        episodes.groupby("HOUSEHOLD_KEY", group_keys=False).apply(_count_for_group)
    )
    return episodes


# --------------------------------------------------------------------------- #
# 3. Transaction audit (Step 3)
# --------------------------------------------------------------------------- #

def audit_transactions(cfg: Config) -> dict:
    """
    Streams the transaction file in chunks (it's ~6M rows) and reports:
      - rows/units carried by extreme-quantity outliers (possible fuel mixing)
      - household-week combinations with zero shopping trips
      - basic negative/zero-value integrity checks
    Does not modify the file; returns a summary dict to inform filtering
    decisions made explicitly later (never silently).
    """
    log.info("Auditing transaction file (chunked)")

    qty_values = []
    total_rows = 0
    total_qty = 0.0
    neg_sales = 0
    neg_qty = 0
    hh_week_pairs = set()
    all_households = set()
    all_weeks = set()

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        total_rows += len(chunk)
        total_qty += chunk["QUANTITY"].sum()
        neg_sales += (chunk["SALES_VALUE"] < 0).sum()
        neg_qty += (chunk["QUANTITY"] < 0).sum()

        qty_values.append(chunk["QUANTITY"].to_numpy())

        pairs = set(zip(chunk["household_key"], chunk["WEEK_NO"]))
        hh_week_pairs |= pairs
        all_households |= set(chunk["household_key"].unique())
        all_weeks |= set(chunk["WEEK_NO"].unique())

    qty_all = np.concatenate(qty_values)
    q99 = np.quantile(qty_all, 0.99)
    outlier_mask = qty_all >= q99
    outlier_row_share = outlier_mask.mean()
    outlier_unit_share = qty_all[outlier_mask].sum() / qty_all.sum()

    n_possible_hh_weeks = len(all_households) * len(all_weeks)
    no_trip_share = 1 - (len(hh_week_pairs) / n_possible_hh_weeks) if n_possible_hh_weeks else np.nan

    summary = {
        "total_rows": total_rows,
        "total_quantity": float(total_qty),
        "negative_sales_rows": int(neg_sales),
        "negative_quantity_rows": int(neg_qty),
        "top_1pct_qty_row_share": float(outlier_row_share),
        "top_1pct_qty_unit_share": float(outlier_unit_share),
        "n_households": len(all_households),
        "n_weeks": len(all_weeks),
        "household_week_no_trip_share": float(no_trip_share),
    }
    log.info(f"Audit summary: {summary}")
    log.warning(
        "top_1pct_qty_unit_share above should be compared against the PDF's claimed "
        "98.7%-units-in-1.2%-of-rows fuel-mixing figure. If far lower, this dataset's "
        "fuel contamination may differ from the PDF's reference dataset -- do not assume "
        "it transfers."
    )
    return summary


# --------------------------------------------------------------------------- #
# 4. Outcome construction (Y columns for the episode table)
# --------------------------------------------------------------------------- #

def compute_household_campaign_outcomes(
    cfg: Config,
    campaign_desc: pd.DataFrame,
    coupon: pd.DataFrame,
    product: pd.DataFrame,
    households_in_scope: set,
) -> pd.DataFrame:
    """
    Streams the transaction file ONCE and computes, for EVERY household in
    households_in_scope x EVERY campaign (not just linked pairs), outcomes
    over that campaign's calendar window:
      Y_ELIGIBLE_SALES / Y_ELIGIBLE_UNITS : sales of THAT campaign's eligible products
      Y_CATEGORY_SALES                    : sales in the SAME COMMODITY as that
                                             campaign's eligible products (not all sales)
      Y_RIVAL_SALES                       : Y_CATEGORY_SALES minus Y_ELIGIBLE_SALES
                                             (same commodity, non-eligible products)
      Y_PRE_SALES / Y_POST_SALES          : household total sales in the pre/post windows

    Computing this for every household (linked or not) against every campaign
    is what makes Plan 1's control group comparable: a control household's
    "during" outcome is now measured over the SAME calendar window as the
    campaign it's being matched against, not over its own unrelated episode.

    30 campaigns x ~2,500 households is a small enough combination to hold in
    memory; only the streaming transaction read needs chunking.
    """
    log.info("Computing household x campaign window outcomes (universal table)")

    # Eligible products per campaign, and the commodities those products belong to.
    eligible = coupon[["CAMPAIGN", "PRODUCT_ID"]].drop_duplicates()
    product_commodity = product.set_index("PRODUCT_ID")["COMMODITY_DESC"]
    eligible = eligible.assign(COMMODITY_DESC=eligible["PRODUCT_ID"].map(product_commodity))

    eligible_products_by_campaign = eligible.groupby("CAMPAIGN")["PRODUCT_ID"].apply(set).to_dict()
    eligible_commodities_by_campaign = (
        eligible.groupby("CAMPAIGN")["COMMODITY_DESC"].apply(lambda s: set(s.dropna())).to_dict()
    )

    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]]
    campaigns = windows.index.tolist()

    accum = {col: {} for col in [
        "Y_ELIGIBLE_SALES", "Y_ELIGIBLE_UNITS", "Y_CATEGORY_SALES",
        "Y_RIVAL_SALES", "Y_PRE_SALES", "Y_POST_SALES",
    ]}

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk = chunk[chunk["household_key"].isin(households_in_scope)]
        if chunk.empty:
            continue
        chunk["COMMODITY_DESC"] = chunk["PRODUCT_ID"].map(product_commodity)

        # Vectorized per-campaign (30 iterations), not per-household (thousands
        # of iterations) -- each iteration is a handful of pandas groupby-sums
        # over the whole chunk rather than one over a tiny per-household slice.
        for campaign in campaigns:
            start, end = windows.loc[campaign, "START_DAY"], windows.loc[campaign, "END_DAY"]
            pre_start = start - cfg.pre_period_weeks * 7
            post_end = end + cfg.post_period_weeks * 7

            elig_products = eligible_products_by_campaign.get(campaign, set())
            elig_commodities = eligible_commodities_by_campaign.get(campaign, set())

            during = chunk[(chunk["DAY"] >= start) & (chunk["DAY"] <= end)]
            pre = chunk[(chunk["DAY"] >= pre_start) & (chunk["DAY"] < start)]
            post = chunk[(chunk["DAY"] > end) & (chunk["DAY"] <= post_end)]

            if len(during):
                is_elig = during["PRODUCT_ID"].isin(elig_products)
                same_commodity = during["COMMODITY_DESC"].isin(elig_commodities)

                elig_sales_by_hh = during.loc[is_elig].groupby("household_key")["SALES_VALUE"].sum()
                elig_units_by_hh = during.loc[is_elig].groupby("household_key")["QUANTITY"].sum()
                category_sales_by_hh = during.loc[same_commodity].groupby("household_key")["SALES_VALUE"].sum()

                for hh, val in elig_sales_by_hh.items():
                    accum["Y_ELIGIBLE_SALES"][(hh, campaign)] = accum["Y_ELIGIBLE_SALES"].get((hh, campaign), 0) + val
                for hh, val in elig_units_by_hh.items():
                    accum["Y_ELIGIBLE_UNITS"][(hh, campaign)] = accum["Y_ELIGIBLE_UNITS"].get((hh, campaign), 0) + val
                for hh, val in category_sales_by_hh.items():
                    key = (hh, campaign)
                    accum["Y_CATEGORY_SALES"][key] = accum["Y_CATEGORY_SALES"].get(key, 0) + val
                    accum["Y_RIVAL_SALES"][key] = accum["Y_RIVAL_SALES"].get(key, 0) + val - elig_sales_by_hh.get(hh, 0)

            if len(pre):
                pre_sales_by_hh = pre.groupby("household_key")["SALES_VALUE"].sum()
                for hh, val in pre_sales_by_hh.items():
                    key = (hh, campaign)
                    accum["Y_PRE_SALES"][key] = accum["Y_PRE_SALES"].get(key, 0) + val
            if len(post):
                post_sales_by_hh = post.groupby("household_key")["SALES_VALUE"].sum()
                for hh, val in post_sales_by_hh.items():
                    key = (hh, campaign)
                    accum["Y_POST_SALES"][key] = accum["Y_POST_SALES"].get(key, 0) + val

    all_keys = set()
    for d in accum.values():
        all_keys |= set(d.keys())
    idx = pd.MultiIndex.from_tuples(sorted(all_keys), names=["HOUSEHOLD_KEY", "CAMPAIGN"])
    out = pd.DataFrame(index=idx)
    for col, d in accum.items():
        out[col] = pd.Series(d)
    out = out.fillna(0.0).reset_index()

    log.info(f"Universal outcome table: {len(out)} household x campaign rows for {len(households_in_scope)} households")
    return out


# --------------------------------------------------------------------------- #
# 5. Plan 1: matched stacked event-study DiD
# --------------------------------------------------------------------------- #

def run_plan1_event_study_did(
    episodes: pd.DataFrame,
    universal_outcomes: pd.DataFrame,
    demographics: pd.DataFrame,
    outcome_col: str = "Y_ELIGIBLE_SALES",
) -> pd.DataFrame:
    """
    For each campaign:
      1. Treated = households assigned to this campaign with N_CONCURRENT_CAMPAIGNS == 0
         (concurrently-treated households are excluded from Plan 1, not modeled --
         Plan 1 can't separate simultaneous treatments).
      2. Controls = every OTHER household in universal_outcomes that is NOT linked
         to this campaign in episodes, regardless of what campaigns they're linked
         to elsewhere -- their outcome is pulled from universal_outcomes for THIS
         campaign's calendar window, so treated and control are compared over the
         identical time period. (Controls linked to a campaign that overlaps this
         one's window are still excluded, since their own treatment would confound
         the comparison.)
      3. Match treated to controls on available demographics + pre-period sales
         (nearest-neighbor on a propensity score from logistic regression).
      4. DiD estimate = (during_treated - pre_treated) - (during_control - pre_control),
         normalized to per-week.
    Returns one row per campaign with the DiD estimate and a naive matched-pair SE
    (a placeholder -- the PDF wants bootstrap/Fieller-style intervals at the ROI
    stage, not this SE; this is enough to rank campaigns directionally, not to
    report as a final interval).
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import NearestNeighbors

    log.info("Running Plan 1: matched stacked event-study DiD")
    results = []

    # household -> set of campaigns they're linked to, for exclusion checks
    linked_campaigns_by_hh = episodes.groupby("HOUSEHOLD_KEY")["CAMPAIGN"].apply(set).to_dict()
    concurrent_free_hh = set(episodes.loc[episodes["N_CONCURRENT_CAMPAIGNS"].eq(0), "HOUSEHOLD_KEY"])

    windows = episodes.drop_duplicates("CAMPAIGN").set_index("CAMPAIGN")[["START_DAY", "END_DAY", "DURATION_DAYS"]]
    all_campaign_windows = windows[["START_DAY", "END_DAY"]]

    demo_cols = [c for c in demographics.columns if c.endswith("_DESC")]
    uo = universal_outcomes.merge(demographics, on="HOUSEHOLD_KEY", how="left")

    for campaign in windows.index:
        c_start, c_end = windows.loc[campaign, ["START_DAY", "END_DAY"]]
        duration_weeks = windows.loc[campaign, "DURATION_DAYS"] / 7.0

        treated_hh = set(episodes.loc[episodes["CAMPAIGN"] == campaign, "HOUSEHOLD_KEY"]) & concurrent_free_hh

        def _overlaps_campaign(other_campaign: int) -> bool:
            o_start, o_end = all_campaign_windows.loc[other_campaign]
            return c_start <= o_end and o_start <= c_end

        overlapping_campaigns = {c for c in all_campaign_windows.index if c != campaign and _overlaps_campaign(c)}

        control_hh = {
            hh for hh, camps in linked_campaigns_by_hh.items()
            if hh not in treated_hh and not (camps & ({campaign} | overlapping_campaigns))
        }
        # Households present in the transaction data but never linked to any campaign are valid controls too.
        control_hh |= set(universal_outcomes["HOUSEHOLD_KEY"].unique()) - set(linked_campaigns_by_hh.keys()) - treated_hh

        treated_rows = uo[(uo["CAMPAIGN"] == campaign) & (uo["HOUSEHOLD_KEY"].isin(treated_hh))].copy()
        control_rows = uo[(uo["CAMPAIGN"] == campaign) & (uo["HOUSEHOLD_KEY"].isin(control_hh))].copy()

        if len(treated_rows) < 10 or len(control_rows) < 10:
            log.info(f"Campaign {campaign}: too few treated/control households after exclusions, skipping direct estimate (needs pooling)")
            results.append({"CAMPAIGN": campaign, "N_TREATED": len(treated_rows), "N_CONTROL": len(control_rows), "DID_ESTIMATE_PER_WEEK": np.nan, "NOTE": "insufficient sample -- requires hierarchical pooling"})
            continue

        combo = pd.concat([treated_rows.assign(_T=1), control_rows.assign(_T=0)], ignore_index=True)
        feature_cols = demo_cols + ["Y_PRE_SALES"]
        X = pd.get_dummies(combo[feature_cols], dummy_na=True).fillna(0)

        try:
            from sklearn.preprocessing import StandardScaler
            X_scaled = StandardScaler().fit_transform(X)
            ps_model = LogisticRegression(max_iter=2000)
            ps_model.fit(X_scaled, combo["_T"])
            combo["_pscore"] = ps_model.predict_proba(X_scaled)[:, 1]
        except Exception as e:
            log.warning(f"Campaign {campaign}: propensity model failed ({e}), using pre-period sales only for matching")
            combo["_pscore"] = combo["Y_PRE_SALES"]

        treated_idx = combo[combo["_T"] == 1].index
        control_idx = combo[combo["_T"] == 0].index
        nn = NearestNeighbors(n_neighbors=1).fit(combo.loc[control_idx, ["_pscore"]])
        _, match_pos = nn.kneighbors(combo.loc[treated_idx, ["_pscore"]])
        matched_control_idx = combo.loc[control_idx].iloc[match_pos.flatten()].index

        treated_during = combo.loc[treated_idx, outcome_col].to_numpy()
        treated_pre = combo.loc[treated_idx, "Y_PRE_SALES"].to_numpy()
        control_during = combo.loc[matched_control_idx, outcome_col].to_numpy()
        control_pre = combo.loc[matched_control_idx, "Y_PRE_SALES"].to_numpy()

        did_per_pair = (treated_during - treated_pre) - (control_during - control_pre)
        did_estimate = np.nanmean(did_per_pair) / max(duration_weeks, 1e-9)
        se = np.nanstd(did_per_pair, ddof=1) / np.sqrt(len(did_per_pair)) / max(duration_weeks, 1e-9)
        post_treated = combo.loc[treated_idx, "Y_POST_SALES"].to_numpy()
        post_control = combo.loc[matched_control_idx, "Y_POST_SALES"].to_numpy()
        post_did_per_pair = (post_treated - treated_pre) - (post_control - control_pre)
        post_did_estimate = np.nanmean(post_did_per_pair) / max(duration_weeks, 1e-9)
        results.append({
            "CAMPAIGN": campaign,
            "N_TREATED": len(treated_rows),
            "N_CONTROL": len(control_rows),
            "DID_ESTIMATE_PER_WEEK": did_estimate,
            "SE_PER_WEEK": se,
            "NOTE": "matched, single campaign -- pool across campaigns before reporting",
        })

    return pd.DataFrame(results)


# --------------------------------------------------------------------------- #
# 6. Hierarchical shrinkage and ranking (PDF Step 9)
# --------------------------------------------------------------------------- #

def pool_campaign_effects(plan1_results: pd.DataFrame, episodes: pd.DataFrame) -> pd.DataFrame:
    """
    Random-effects (DerSimonian-Laird) partial pooling of the per-campaign DiD
    estimates, nested by campaign type (TypeA/B/C from campaign_desc's
    DESCRIPTION field), addressing two problems the raw Plan 1 output has:

      1. Campaigns with too few treated/control households (DID_ESTIMATE_PER_WEEK
         is NaN) get no estimate at all in the raw output. Here they're assigned
         the pooled mean of same-type campaigns that DID get a direct estimate,
         falling back to the grand mean if their type has none either.
      2. Campaigns WITH a direct estimate are shrunk toward their type's pooled
         mean in proportion to how noisy they are (small SE -> little shrinkage,
         large SE -> shrunk hard toward the group). This is what the PDF calls
         "shrink noisy estimates toward the overall mean" and is a first step
         toward correcting the winner's curse -- it is NOT the full
         selection-corrected expectation the PDF asks for at final reporting
         (that needs a proper post-selection inference step, not implemented
         here), but it stops small campaigns from ranking artificially high.

    Returns plan1_results with added columns: CAMPAIGN_TYPE, GROUP_MEAN,
    TAU2 (between-campaign variance within type), SHRUNK_ESTIMATE_PER_WEEK,
    IS_DIRECT_ESTIMATE (0 if this campaign had no usable sample and is fully
    pooled), and RANK (by SHRUNK_ESTIMATE_PER_WEEK, descending).
    """
    log.info("Pooling campaign effects (empirical-Bayes shrinkage by campaign type)")

    campaign_type = episodes.drop_duplicates("CAMPAIGN").set_index("CAMPAIGN")["CAMPAIGN_TYPE"]
    df = plan1_results.copy()
    df["CAMPAIGN_TYPE"] = df["CAMPAIGN"].map(campaign_type)
    df["IS_DIRECT_ESTIMATE"] = df["DID_ESTIMATE_PER_WEEK"].notna().astype(int)

    def _pool_group(group: pd.DataFrame) -> pd.DataFrame:
        direct = group[group["IS_DIRECT_ESTIMATE"] == 1]
        if len(direct) == 0:
            group["GROUP_MEAN"] = np.nan
            group["TAU2"] = np.nan
            group["SHRUNK_ESTIMATE_PER_WEEK"] = np.nan
            return group

        y = direct["DID_ESTIMATE_PER_WEEK"].to_numpy()
        se = direct["SE_PER_WEEK"].replace(0, np.nan).to_numpy()
        se = np.where(np.isnan(se), np.nanmedian(se) if not np.all(np.isnan(se)) else 1.0, se)
        w = 1.0 / (se ** 2)

        y_bar = np.sum(w * y) / np.sum(w)
        if len(y) > 1:
            Q = np.sum(w * (y - y_bar) ** 2)
            C = np.sum(w) - np.sum(w ** 2) / np.sum(w)
            tau2 = max(0.0, (Q - (len(y) - 1)) / C) if C > 0 else 0.0
        else:
            tau2 = 0.0  # single-campaign type: no between-campaign variance to estimate

        group["GROUP_MEAN"] = y_bar
        group["TAU2"] = tau2

        def _shrink(row):
            if row["IS_DIRECT_ESTIMATE"] == 0:
                return y_bar  # no direct estimate: fully pooled to group mean
            se_i2 = row["SE_PER_WEEK"] ** 2 if pd.notna(row["SE_PER_WEEK"]) and row["SE_PER_WEEK"] > 0 else np.nanmedian(se) ** 2
            if tau2 == 0:
                return y_bar
            lam = tau2 / (tau2 + se_i2)  # weight on the campaign's own estimate
            return lam * row["DID_ESTIMATE_PER_WEEK"] + (1 - lam) * y_bar

        group["SHRUNK_ESTIMATE_PER_WEEK"] = group.apply(_shrink, axis=1)
        return group

    pooled = df.groupby("CAMPAIGN_TYPE", group_keys=False).apply(_pool_group)

    # Fallback: any campaign type with zero direct estimates in it gets the
    # grand mean across ALL campaigns with direct estimates, not left as NaN.
    grand_mean = df.loc[df["IS_DIRECT_ESTIMATE"] == 1, "DID_ESTIMATE_PER_WEEK"].mean()
    pooled["SHRUNK_ESTIMATE_PER_WEEK"] = pooled["SHRUNK_ESTIMATE_PER_WEEK"].fillna(grand_mean)
    pooled["GROUP_MEAN"] = pooled["GROUP_MEAN"].fillna(grand_mean)

    pooled = pooled.sort_values("SHRUNK_ESTIMATE_PER_WEEK", ascending=False).reset_index(drop=True)
    pooled["RANK"] = pooled.index + 1

    log.info(
        f"Pooling complete: {pooled['IS_DIRECT_ESTIMATE'].sum()} campaigns had a direct "
        f"estimate, {len(pooled) - pooled['IS_DIRECT_ESTIMATE'].sum()} were fully pooled "
        f"from their type group or the grand mean."
    )
    return pooled


# --------------------------------------------------------------------------- #
# 7. Placebo test (PDF Step 6 -- validate the counterfactual)
# --------------------------------------------------------------------------- #
# def run_placebo_test_iptw(
#     episodes, universal_outcomes, pooled_results,
#     outcome_col="Y_ELIGIBLE_SALES", n_draws_per_campaign=200, seed=42,
# ):
#     """
#     Same null-distribution logic as run_placebo_test, but computes each draw's
#     treated/control means as IPTW-weighted means (propensity from PRE_SALES,
#     same spec as execute_doubly_robust_pipeline) instead of unweighted means.
#     If propensity adjustment is doing its job, the 20/30 pre-period failures
#     from the naive version should shrink here. If they don't, the model needs
#     more covariates before TRUE_CAUSAL_LIFT can be trusted.
#     """
#     import statsmodels.formula.api as smf

#     log.info(f"Running IPTW-weighted placebo test: {n_draws_per_campaign} draws per campaign")
#     rng = np.random.default_rng(seed)

#     linked_hh = set(episodes["HOUSEHOLD_KEY"].unique())
#     never_linked_hh = np.array(sorted(set(universal_outcomes["HOUSEHOLD_KEY"].unique()) - linked_hh))

#     duration_weeks_by_campaign = (
#         episodes.drop_duplicates("CAMPAIGN").set_index("CAMPAIGN")["DURATION_DAYS"] / 7.0
#     )

#     draws = []
#     for _, row in pooled_results.drop_duplicates("CAMPAIGN").iterrows():
#         campaign = row["CAMPAIGN"]
#         n_treated = int(row["N_TREATED"]) if pd.notna(row["N_TREATED"]) and row["N_TREATED"] > 0 else 30
#         duration_weeks = duration_weeks_by_campaign.get(campaign, 4.0)

#         pool = universal_outcomes[
#             (universal_outcomes["CAMPAIGN"] == campaign)
#             & (universal_outcomes["HOUSEHOLD_KEY"].isin(never_linked_hh))
#         ].copy()
#         if len(pool) < 20:
#             continue

#         n_treated_draw = min(n_treated, len(pool) // 2)

#         for _ in range(n_draws_per_campaign):
#             shuffled = rng.permutation(pool["HOUSEHOLD_KEY"].to_numpy())
#             fake_treated_hh = set(shuffled[:n_treated_draw])

#             draw = pool.copy()
#             draw["_T"] = draw["HOUSEHOLD_KEY"].isin(fake_treated_hh).astype(int)
#             draw["_PRE_ADJ"] = draw["Y_PRE_SALES"] + 0.001

#             try:
#                 logit = smf.logit("_T ~ _PRE_ADJ", data=draw).fit(disp=False)
#                 draw["_PS"] = logit.predict(draw).clip(0.001, 0.999)
#                 draw["_W"] = np.where(draw["_T"] == 1, 1.0, draw["_PS"] / (1 - draw["_PS"]))
#                 cap = draw.loc[draw["_T"] == 0, "_W"].quantile(0.99)
#                 draw.loc[draw["_T"] == 0, "_W"] = draw.loc[draw["_T"] == 0, "_W"].clip(upper=cap)
#             except Exception:
#                 continue  # skip draws where the tiny logit fails to converge

#             t = draw[draw["_T"] == 1]
#             c = draw[draw["_T"] == 0]
#             t_delta = np.average(t[outcome_col] - t["Y_PRE_SALES"], weights=t["_W"])
#             c_delta = np.average(c[outcome_col] - c["Y_PRE_SALES"], weights=c["_W"])
#             placebo_did = (t_delta - c_delta) / max(duration_weeks, 1e-9)

#             draws.append({"CAMPAIGN": campaign, "PLACEBO_DID_PER_WEEK": placebo_did})

#     placebo_draws_iptw = pd.DataFrame(draws)
#     if placebo_draws_iptw.empty:
#         log.warning("IPTW placebo test produced no draws.")
#         pooled_results["EMPIRICAL_P_VALUE_IPTW"] = np.nan
#         return placebo_draws_iptw, pooled_results

#     null_values = placebo_draws_iptw["PLACEBO_DID_PER_WEEK"].to_numpy()
#     log.info(f"IPTW placebo null: n={len(null_values)}, mean={null_values.mean():.4f}, sd={null_values.std():.4f}")
def run_placebo_test(
    episodes: pd.DataFrame,
    universal_outcomes: pd.DataFrame,
    pooled_results: pd.DataFrame,
    outcome_col: str = "Y_ELIGIBLE_SALES",
    n_draws_per_campaign: int = 200,
    seed: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Assigns FAKE treatment within the pool of households that were NEVER linked
    to ANY real campaign, using each real campaign's actual window and sample
    size, and computes the same DiD statistic. Repeated many times per campaign.
    Since no real treatment exists in this pool, the resulting distribution is
    a null: what DID_ESTIMATE_PER_WEEK values arise from pure sampling noise
    given this data's structure.

    This does NOT re-run the propensity-score matching from Plan 1 (that would
    be n_draws_per_campaign x 30 x logistic-regression-fits -- too slow for
    what's needed here). It uses a simple mean-difference DiD on random splits
    instead, which is the right tool for characterizing a null distribution
    (matching mainly helps point estimates, not the noise floor).

    Returns:
      placebo_draws       : every individual placebo draw (for inspection/plotting)
      pooled_with_pvalues : pooled_results with an added EMPIRICAL_P_VALUE column,
                             the two-sided share of ALL placebo draws (pooled
                             across campaigns, for a stabler tail estimate) at
                             least as extreme as that campaign's shrunk estimate.
    """
    log.info(f"Running placebo test: {n_draws_per_campaign} draws per campaign on never-linked households")
    rng = np.random.default_rng(seed)

    linked_hh = set(episodes["HOUSEHOLD_KEY"].unique())
    never_linked_hh = np.array(sorted(set(universal_outcomes["HOUSEHOLD_KEY"].unique()) - linked_hh))
    log.info(f"{len(never_linked_hh)} households were never linked to any real campaign -- placebo pool")

    duration_weeks_by_campaign = (
        episodes.drop_duplicates("CAMPAIGN").set_index("CAMPAIGN")["DURATION_DAYS"] / 7.0
    )

    draws = []
    for _, row in pooled_results.drop_duplicates("CAMPAIGN").iterrows():
        campaign = row["CAMPAIGN"]
        n_treated = int(row["N_TREATED"]) if pd.notna(row["N_TREATED"]) and row["N_TREATED"] > 0 else 30
        duration_weeks = duration_weeks_by_campaign.get(campaign, 4.0)

        pool = universal_outcomes[
            (universal_outcomes["CAMPAIGN"] == campaign)
            & (universal_outcomes["HOUSEHOLD_KEY"].isin(never_linked_hh))
        ]
        if len(pool) < 20:
            continue  # not enough never-linked households with data for this campaign to draw from

        n_treated_draw = min(n_treated, len(pool) // 2)

        for _ in range(n_draws_per_campaign):
            shuffled = rng.permutation(pool["HOUSEHOLD_KEY"].to_numpy())
            fake_treated_hh = set(shuffled[:n_treated_draw])
            fake_control_hh = set(shuffled[n_treated_draw:])

            ft = pool[pool["HOUSEHOLD_KEY"].isin(fake_treated_hh)]
            fc = pool[pool["HOUSEHOLD_KEY"].isin(fake_control_hh)]

            treated_delta = ft[outcome_col].mean() - ft["Y_PRE_SALES"].mean()
            control_delta = fc[outcome_col].mean() - fc["Y_PRE_SALES"].mean()
            placebo_did = (treated_delta - control_delta) / max(duration_weeks, 1e-9)

            draws.append({"CAMPAIGN": campaign, "PLACEBO_DID_PER_WEEK": placebo_did})

    placebo_draws = pd.DataFrame(draws)
    if placebo_draws.empty:
        log.warning("Placebo test produced no draws -- likely too few never-linked households in the transaction file.")
        pooled_results["EMPIRICAL_P_VALUE"] = np.nan
        return placebo_draws, pooled_results

    null_values = placebo_draws["PLACEBO_DID_PER_WEEK"].to_numpy()
    log.info(
        f"Placebo null distribution: n={len(null_values)}, mean={null_values.mean():.4f}, "
        f"sd={null_values.std():.4f} (mean should be near 0 if the method is unbiased)"
    )

    def _p_value(estimate):
        if pd.isna(estimate):
            return np.nan
        return float(np.mean(np.abs(null_values) >= np.abs(estimate)))

    pooled_with_pvalues = pooled_results.copy()
    pooled_with_pvalues["EMPIRICAL_P_VALUE"] = pooled_with_pvalues["SHRUNK_ESTIMATE_PER_WEEK"].apply(_p_value)

    return placebo_draws, pooled_with_pvalues
def run_placebo_test_iptw(
    episodes, universal_outcomes, pooled_results, hh_demographic,
    outcome_col="Y_ELIGIBLE_SALES", n_draws_per_campaign=200, seed=42,
):
    """
    Same as before, but propensity model is now sklearn's LogisticRegression
    (L2-regularized) instead of statsmodels' Logit/fit_regularized -- sklearn's
    solver is numerically stable under separation by construction (bounded
    coefficient updates), so it doesn't hit the overflow/divide-by-zero warnings
    statsmodels' optimizer produces on its search path with ~7 demographic
    dummies and small per-draw samples. Functionally equivalent propensity
    scores, no numerical noise.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    log.info(f"Running IPTW-weighted placebo test (demographics-adjusted, sklearn): {n_draws_per_campaign} draws per campaign")
    rng = np.random.default_rng(seed)

    linked_hh = set(episodes["HOUSEHOLD_KEY"].unique())
    never_linked_hh = np.array(sorted(set(universal_outcomes["HOUSEHOLD_KEY"].unique()) - linked_hh))

    duration_weeks_by_campaign = (
        episodes.drop_duplicates("CAMPAIGN").set_index("CAMPAIGN")["DURATION_DAYS"] / 7.0
    )

    demo_cols = ["AGE_DESC", "MARITAL_STATUS_CODE", "INCOME_DESC", "HOMEOWNER_DESC",
                 "HH_COMP_DESC", "HOUSEHOLD_SIZE_DESC", "KID_CATEGORY_DESC"]
    demo = hh_demographic[["HOUSEHOLD_KEY"] + demo_cols].drop_duplicates("HOUSEHOLD_KEY")

    draws = []
    n_attempted = 0
    n_failed = 0

    for _, row in pooled_results.drop_duplicates("CAMPAIGN").iterrows():
        campaign = row["CAMPAIGN"]
        n_treated = int(row["N_TREATED"]) if pd.notna(row["N_TREATED"]) and row["N_TREATED"] > 0 else 30
        duration_weeks = duration_weeks_by_campaign.get(campaign, 4.0)

        pool = universal_outcomes[
            (universal_outcomes["CAMPAIGN"] == campaign)
            & (universal_outcomes["HOUSEHOLD_KEY"].isin(never_linked_hh))
        ].merge(demo, on="HOUSEHOLD_KEY", how="left")
        if len(pool) < 20:
            continue

        n_treated_draw = min(n_treated, len(pool) // 2)
        X_demo = pd.get_dummies(pool[demo_cols], dummy_na=True).astype(float)

        for _ in range(n_draws_per_campaign):
            n_attempted += 1
            shuffled = rng.permutation(pool["HOUSEHOLD_KEY"].to_numpy())
            fake_treated_hh = set(shuffled[:n_treated_draw])

            draw = pool.copy()
            draw["_T"] = draw["HOUSEHOLD_KEY"].isin(fake_treated_hh).astype(int)
            X = X_demo.copy()
            X["_PRE_ADJ"] = draw["Y_PRE_SALES"].to_numpy() + 0.001

            if draw["_T"].nunique() < 2:
                continue

            try:
                X_scaled = StandardScaler().fit_transform(X)
                ps_model = LogisticRegression(penalty="l2", C=1.0, max_iter=1000)
                ps_model.fit(X_scaled, draw["_T"])
                draw["_PS"] = np.clip(ps_model.predict_proba(X_scaled)[:, 1], 0.001, 0.999)
                draw["_W"] = np.where(draw["_T"] == 1, 1.0, draw["_PS"] / (1 - draw["_PS"]))
                cap = draw.loc[draw["_T"] == 0, "_W"].quantile(0.99)
                draw.loc[draw["_T"] == 0, "_W"] = draw.loc[draw["_T"] == 0, "_W"].clip(upper=cap)
            except Exception:
                n_failed += 1
                continue

            t = draw[draw["_T"] == 1]
            c = draw[draw["_T"] == 0]
            t_delta = np.average(t[outcome_col] - t["Y_PRE_SALES"], weights=t["_W"])
            c_delta = np.average(c[outcome_col] - c["Y_PRE_SALES"], weights=c["_W"])
            placebo_did = (t_delta - c_delta) / max(duration_weeks, 1e-9)

            draws.append({"CAMPAIGN": campaign, "PLACEBO_DID_PER_WEEK": placebo_did})

    log.info(f"Draws attempted: {n_attempted}, failed/skipped: {n_failed} ({n_failed/max(n_attempted,1):.1%}), survived: {len(draws)}")

    placebo_draws_iptw = pd.DataFrame(draws)
    if placebo_draws_iptw.empty:
        log.warning("IPTW placebo test produced no draws.")
        pooled_results["EMPIRICAL_P_VALUE_IPTW"] = np.nan
        return placebo_draws_iptw, pooled_results

    null_values = placebo_draws_iptw["PLACEBO_DID_PER_WEEK"].to_numpy()
    log.info(f"IPTW (demo-adjusted, sklearn) placebo null: n={len(null_values)}, mean={null_values.mean():.4f}, sd={null_values.std():.4f}")

    def _p_value(estimate):
        return np.nan if pd.isna(estimate) else float(np.mean(np.abs(null_values) >= np.abs(estimate)))

    out = pooled_results.copy()
    out["EMPIRICAL_P_VALUE_IPTW"] = out["SHRUNK_ESTIMATE_PER_WEEK"].apply(_p_value)
    return placebo_draws_iptw, out
    
# --------------------------------------------------------------------------- #
# 6. Orchestration
# --------------------------------------------------------------------------- #
def run_pipeline(cfg: Config) -> dict:
    tables = load_reference_tables(cfg)
    episodes = build_episode_table(tables)
    audit_summary = audit_transactions(cfg)

    universal_outcomes = build_universal_outcomes(
        cfg, tables, post_windows_by_campaign=campaign_post_windows_v3
    )
    # print(episodes.columns)

    universal_outcomes = universal_outcomes.rename(columns={"household_key": "HOUSEHOLD_KEY"})
    episodes_with_outcomes = episodes.merge(
        universal_outcomes, on=["HOUSEHOLD_KEY", "CAMPAIGN"], how="left"
    )
    outcome_cols = [c for c in universal_outcomes.columns if c.startswith("Y_")]
    episodes_with_outcomes[outcome_cols] = episodes_with_outcomes[outcome_cols].fillna(0.0)

    plan1_results = run_plan1_event_study_did(episodes, universal_outcomes, tables["hh_demographic"])
    pooled_results = pool_campaign_effects(plan1_results, episodes)

    # naive version too, for the direct before/after comparison
    placebo_draws_iptw, pooled_results = run_placebo_test_iptw(
    episodes, universal_outcomes, pooled_results, tables["hh_demographic"])
    n_iptw_failures = int((pooled_results["EMPIRICAL_P_VALUE_IPTW"] < 0.05).sum())
    log.info(f"IPTW (demo-adjusted) placebo test: {n_iptw_failures}/30 campaigns still significant")

    placebo_draws_naive, pooled_results = run_placebo_test(episodes, universal_outcomes, pooled_results)
    n_naive_failures = int((pooled_results["EMPIRICAL_P_VALUE"] < 0.05).sum())
    log.info(f"Naive placebo test: {n_naive_failures}/30 campaigns still show significant pre-period differences")

    return {
        "episodes": episodes_with_outcomes,
        "audit_summary": audit_summary,
        "plan1_results": plan1_results,
        "pooled_results": pooled_results,
        "placebo_draws_iptw": placebo_draws_iptw,
        "placebo_draws_naive": placebo_draws_naive,
    }

def main(data_dir: str, out_dir: str = "./pipeline_output"):
    """Callable directly from a notebook: main('/path/to/csvs', './pipeline_output')"""
    cfg = Config(data_dir=Path(data_dir))
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    outputs = run_pipeline(cfg)
    outputs["episodes"].to_csv(out_path / "episode_table_with_outcomes.csv", index=False)
    outputs["plan1_results"].to_csv(out_path / "plan1_did_results.csv", index=False)
    outputs["pooled_results"].to_csv(out_path / "pooled_campaign_effects.csv", index=False)
    outputs["placebo_draws_iptw"].to_csv(out_path / "placebo_null_distribution.csv", index=False)
    outputs["placebo_draws_naive"].to_csv(out_path / "placebo_null_distribution.csv", index=False)

    log.info(f"Done. Outputs written to {out_path}")
    return outputs


if __name__ == "__main__":
    import sys
    import argparse

    # Jupyter/IPython injects its own kernel-connection args into sys.argv,
    # which argparse chokes on. Detect that case and fall back to editable
    # variables below instead of forcing you to run this as a .py script.
    running_in_notebook = "ipykernel_launcher" in sys.argv[0] or "ipykernel" in sys.modules

    if running_in_notebook:
        log.info("Detected notebook environment -- skipping argparse. Edit DATA_DIR/OUT_DIR below and re-run this cell.")
        DATA_DIR = "data"
        OUT_DIR = "./pipeline_output"
        outputs = main(DATA_DIR, OUT_DIR)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--data-dir", type=str, required=True, help="Directory containing all 7 CSVs")
        parser.add_argument("--out-dir", type=str, default="./pipeline_output")
        args = parser.parse_args()
        outputs = main(args.data_dir, args.out_dir)

    pooled_results = outputs["pooled_results"]
    episodes = outputs["episodes"]  # if you need it downstream too

2026-08-07 10:11:11,556 | INFO | Detected notebook environment -- skipping argparse. Edit DATA_DIR/OUT_DIR below and re-run this cell.
2026-08-07 10:11:11,557 | INFO | Loading reference tables
2026-08-07 10:11:11,628 | INFO | Building episode table
2026-08-07 10:11:11,634 | INFO | coupon.csv: dropped 5164 exact-duplicate rows (4.15%)
2026-08-07 10:11:12,576 | INFO | Episode table: 7208 rows, 1584 households
2026-08-07 10:11:12,577 | INFO | Auditing transaction file (chunked)
2026-08-07 10:11:13,882 | INFO | Audit summary: {'total_rows': 2595732, 'total_quantity': 260685622.0, 'negative_sales_rows': 0, 'negative_quantity_rows': 0, 'top_1pct_qty_row_share': 0.010642470023869952, 'top_1pct_qty_unit_share': 0.9873510054958076, 'n_households': 2500, 'n_weeks': 102, 'household_week_no_trip_share': 0.5138196078431372}
2026-08-07 10:11:13,883 | WARNING | top_1pct_qty_unit_share above should be compared against the PDF's claimed 98.7%-units-in-1.2%-of-rows fuel-mixing figure. If far lower, this

In [202]:
# --------------------------------------------------------------------------- #
# Step 5: Base Price Recovery (p^0), Break-Even Screening (kappa*), & Y_margin
# --------------------------------------------------------------------------- #
import os
import logging
import numpy as np
import pandas as pd

log = logging.getLogger("campaign_pipeline")

def recover_base_prices_rolling_mode(cfg, window_weeks: int = 8, min_obs: int = 3) -> pd.DataFrame:
    """Reconstructs regular shelf prices (p^0) via rolling mode estimation."""
    log.info("Recovering base prices (p^0) via rolling mode estimation...")
    weekly_prices = []
    
    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        valid = chunk[(chunk["QUANTITY"] > 0) & (chunk["SALES_VALUE"] > 0)].copy()
        valid["UNIT_PRICE"] = valid["SALES_VALUE"] / valid["QUANTITY"]
        grp = valid.groupby(["PRODUCT_ID", "WEEK_NO"])["UNIT_PRICE"].median().reset_index()
        weekly_prices.append(grp)
        
    df_prices = pd.concat(weekly_prices, ignore_index=True)
    df_prices = df_prices.groupby(["PRODUCT_ID", "WEEK_NO"])["UNIT_PRICE"].median().reset_index()
    
    df_prices = df_prices.sort_values(["PRODUCT_ID", "WEEK_NO"])
    df_prices["BASE_PRICE_P0"] = (
        df_prices.groupby("PRODUCT_ID")["UNIT_PRICE"]
        .transform(lambda s: s.rolling(window=window_weeks, min_periods=min_obs).max())
    )
    df_prices["BASE_PRICE_P0"] = df_prices.groupby("PRODUCT_ID")["BASE_PRICE_P0"].ffill().bfill()
    
    log.info(f"Base price recovery complete for {df_prices['PRODUCT_ID'].nunique()} products.")
    return df_prices


def compute_campaign_margin_economics(
    cfg, campaign_desc: pd.DataFrame, coupon: pd.DataFrame, base_prices: pd.DataFrame, standard_gross_margin: float = 0.30
) -> pd.DataFrame:
    """Computes average discount depth (d) and break-even incremental share (kappa*)."""
    log.info("Computing campaign discount depths and break-even thresholds (kappa*)...")
    coupon_dedup = coupon.drop_duplicates(subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"])
    eligible = coupon_dedup[["CAMPAIGN", "PRODUCT_ID"]].drop_duplicates()
    
    campaign_weeks = campaign_desc.copy()
    campaign_weeks["START_WEEK"] = (campaign_weeks["START_DAY"] // 7) + 1
    
    merged = eligible.merge(campaign_weeks[["CAMPAIGN", "START_WEEK"]], on="CAMPAIGN", how="left")
    merged = merged.merge(base_prices, left_on=["PRODUCT_ID", "START_WEEK"], right_on=["PRODUCT_ID", "WEEK_NO"], how="left")
    
    during_paid = []
    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]]
    
    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk = chunk[chunk["QUANTITY"] > 0]
        
        for campaign, (start, end) in windows.iterrows():
            sub = chunk[(chunk["DAY"] >= start) & (chunk["DAY"] <= end)]
            if sub.empty:
                continue
            
            elig_skus = set(eligible.loc[eligible["CAMPAIGN"] == campaign, "PRODUCT_ID"])
            sub_elig = sub[sub["PRODUCT_ID"].isin(elig_skus)]
            if not sub_elig.empty:
                during_paid.append({
                    "CAMPAIGN": campaign, 
                    "PAID_SALES": sub_elig["SALES_VALUE"].sum(), 
                    "PAID_QTY": sub_elig["QUANTITY"].sum()
                })
                
    df_paid = pd.DataFrame(during_paid).groupby("CAMPAIGN").sum().reset_index()
    df_paid["P_PAID"] = df_paid["PAID_SALES"] / df_paid["PAID_QTY"]
    
    camp_base = merged.groupby("CAMPAIGN")["BASE_PRICE_P0"].mean().reset_index()
    
    econ = campaign_weeks[["CAMPAIGN", "DESCRIPTION"]].merge(camp_base, on="CAMPAIGN", how="left")
    econ = econ.merge(df_paid[["CAMPAIGN", "P_PAID"]], on="CAMPAIGN", how="left")
    
    econ["DISCOUNT_DEPTH_D"] = (econ["BASE_PRICE_P0"] - econ["P_PAID"]).clip(lower=0)
    econ["GROSS_MARGIN_M"] = standard_gross_margin
    econ["BREAK_EVEN_KAPPA_STAR"] = econ["DISCOUNT_DEPTH_D"] / econ["GROSS_MARGIN_M"]
    econ["UNPROFITABLE_BY_DESIGN"] = econ["BREAK_EVEN_KAPPA_STAR"] >= 1.0
    
    log.info(f"Margin Economics: {econ['UNPROFITABLE_BY_DESIGN'].sum()}/{len(econ)} campaigns unviable by design.")
    return econ


def append_margin_outcomes(universal_outcomes: pd.DataFrame, margin_econ: pd.DataFrame) -> pd.DataFrame:
    """Calculates multi-metric net financial outcome vector Y_margin."""
    log.info("Calculating multi-metric outcome vector (Y_margin)...")
    df = universal_outcomes.merge(margin_econ[["CAMPAIGN", "DISCOUNT_DEPTH_D", "GROSS_MARGIN_M"]], on="CAMPAIGN", how="left")
    
    df["PROFIT_PROMOTED"] = (df["Y_ELIGIBLE_SALES"] * df["GROSS_MARGIN_M"]) - (df["Y_ELIGIBLE_UNITS"] * df["DISCOUNT_DEPTH_D"].fillna(0))
    df["LOST_MARGIN_RIVAL"] = df["Y_RIVAL_SALES"] * df["GROSS_MARGIN_M"]
    df["PAYBACK_COST_POST"] = df["Y_POST_SALES"] * df["GROSS_MARGIN_M"]
    
    # Net Financial Outcome Vector Y_MARGIN
    df["Y_MARGIN"] = df["PROFIT_PROMOTED"] - df["LOST_MARGIN_RIVAL"] - df["PAYBACK_COST_POST"]
    return df


# --------------------------------------------------------------------------- #
# Step 5 Execution
# --------------------------------------------------------------------------- #
base_prices = recover_base_prices_rolling_mode(cfg)

margin_econ = compute_campaign_margin_economics(
    cfg, tables["campaign_desc"], tables["coupon"], base_prices, standard_gross_margin=0.30
)
margin_econ.to_csv("pipeline_output/campaign_margin_economics.csv", index=False)

if "universal_outcomes" in globals():
    df_outcomes = universal_outcomes
elif os.path.exists("pipeline_output/universal_outcomes.csv"):
    df_outcomes = pd.read_csv("pipeline_output/universal_outcomes.csv")
else:
    raise FileNotFoundError("Could not find `universal_outcomes`. Ensure Step 4 is executed first.")

universal_outcomes_with_margin = append_margin_outcomes(df_outcomes, margin_econ)
universal_outcomes_with_margin.to_csv("pipeline_output/universal_outcomes_with_margin.csv", index=False)

margin_econ[[
    "CAMPAIGN", "DESCRIPTION", "BASE_PRICE_P0", "P_PAID", 
    "DISCOUNT_DEPTH_D", "BREAK_EVEN_KAPPA_STAR", "UNPROFITABLE_BY_DESIGN"
]].head(10)

2026-08-07 10:11:54,906 | INFO | Recovering base prices (p^0) via rolling mode estimation...
2026-08-07 10:12:01,335 | INFO | Base price recovery complete for 91905 products.
2026-08-07 10:12:01,338 | INFO | Computing campaign discount depths and break-even thresholds (kappa*)...
2026-08-07 10:12:02,781 | INFO | Margin Economics: 28/30 campaigns unviable by design.
2026-08-07 10:12:02,787 | INFO | Calculating multi-metric outcome vector (Y_margin)...


,CAMPAIGN,DESCRIPTION,BASE_PRICE_P0,P_PAID,DISCOUNT_DEPTH_D,BREAK_EVEN_KAPPA_STAR,UNPROFITABLE_BY_DESIGN
0,24,TypeB,3.580377,2.265161,1.315216,4.384052,True
1,15,TypeC,1.480278,0.668521,0.811757,2.705855,True
2,25,TypeB,3.277935,2.188071,1.089864,3.632880,True
3,20,TypeC,2.708736,2.025208,0.683527,2.278424,True
4,23,TypeB,3.233125,2.228397,1.004728,3.349092,True
5,21,TypeB,3.404000,2.922904,0.481096,1.603652,True
6,22,TypeB,2.864137,2.146961,0.717177,2.390588,True
7,18,TypeA,3.645935,2.497705,1.148230,3.827432,True
8,19,TypeB,4.558598,3.009264,1.549335,5.164449,True
9,17,TypeB,3.250534,2.386731,0.863803,2.879344,True


In [203]:
# --------------------------------------------------------------------------- #
# Step 7: Doubly Robust Causal Lift Estimation (IPTW + ANCOVA)
# --------------------------------------------------------------------------- #
import numpy as np
import pandas as pd
import logging
import statsmodels.formula.api as smf
from collections import defaultdict

log = logging.getLogger("campaign_pipeline")

def build_doubly_robust_dataset(cfg, campaign_desc, tables, pre_days=28):
    log.info("Constructing Pre-Campaign Baseline and Outcome matrix for all households...")

    coupon = tables["coupon"]
    coupon_dedup = coupon.drop_duplicates(subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"])
    elig_map = coupon_dedup.groupby("CAMPAIGN")["PRODUCT_ID"].apply(set).to_dict()

    campaign_table = tables["campaign_table"]
    hh_col = "household_key" if "household_key" in campaign_table.columns else "HOUSEHOLD_KEY"
    if hh_col not in campaign_table.columns:
        hh_col = [c for c in campaign_table.columns if "house" in c.lower()][0]

    treated_map = campaign_table.groupby("CAMPAIGN")[hh_col].apply(set).to_dict()
    hh_campaigns = campaign_table.groupby(hh_col)["CAMPAIGN"].apply(set).to_dict()  # NEW
    universe_hhs = set(campaign_table[hh_col].unique())

    # NEW: precompute, for every campaign, the set of OTHER campaigns whose
    # [START_DAY, END_DAY] window overlaps it -- same rule as Plan 1 and the
    # contamination-exclusion rule in document 3, section 6.
    windows_df = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]]

    def _overlap_set(camp_id):
        s0, e0 = windows_df.loc[camp_id, "START_DAY"], windows_df.loc[camp_id, "END_DAY"]
        others = windows_df.drop(index=camp_id)
        mask = (others["START_DAY"] <= e0) & (s0 <= others["END_DAY"])
        return set(others.index[mask])

    overlap_cache = {c: _overlap_set(c) for c in windows_df.index}

    data = defaultdict(lambda: defaultdict(lambda: [0.0, 0.0]))
    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]].to_dict("index")

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk = chunk[chunk["QUANTITY"] > 0]
        txn_hh_col = "household_key" if "household_key" in chunk.columns else "HOUSEHOLD_KEY"
        chunk = chunk[chunk[txn_hh_col].isin(universe_hhs)]

        for camp_id, bounds in windows.items():
            pre_start, camp_start, camp_end = bounds["START_DAY"] - pre_days, bounds["START_DAY"], bounds["END_DAY"]
            sub = chunk[(chunk["DAY"] >= pre_start) & (chunk["DAY"] <= camp_end)]
            if sub.empty:
                continue
            elig_skus = elig_map.get(camp_id, set())
            sub_elig = sub[sub["PRODUCT_ID"].isin(elig_skus)]
            if sub_elig.empty:
                continue

            pre_df = sub_elig[sub_elig["DAY"] < camp_start]
            for hh, val in pre_df.groupby(txn_hh_col)["SALES_VALUE"].sum().items():
                data[camp_id][hh][0] += val
            camp_df = sub_elig[sub_elig["DAY"] >= camp_start]
            for hh, val in camp_df.groupby(txn_hh_col)["SALES_VALUE"].sum().items():
                data[camp_id][hh][1] += val

    records = []
    n_excluded_total = 0
    for camp_id, hh_dict in data.items():
        treated_hhs = treated_map.get(camp_id, set())
        overlap_camps = overlap_cache.get(camp_id, set())
        n_excluded = 0

        for hh in universe_hhs:
            if hh in treated_hhs:
                is_treated = 1
            else:
                # NEW: contamination exclusion -- a household linked to any
                # campaign whose window overlaps this one is dropped entirely,
                # not silently kept as a control.
                if hh_campaigns.get(hh, set()) & overlap_camps:
                    n_excluded += 1
                    continue
                is_treated = 0

            pre_sales, camp_sales = hh_dict.get(hh, [0.0, 0.0])
            records.append({
                "CAMPAIGN": camp_id, "household_key": hh, "TREATED": is_treated,
                "PRE_SALES": pre_sales, "CAMP_SALES": camp_sales,
            })
        n_excluded_total += n_excluded
        if n_excluded:
            log.info(f"Campaign {camp_id}: excluded {n_excluded} contaminated households from control pool")

    log.info(f"Contamination exclusion removed {n_excluded_total} household-campaign rows total")
    df_dr = pd.DataFrame(records)
    log.info(f"Matrix built successfully. Total shape: {df_dr.shape}")
    return df_dr


def execute_doubly_robust_pipeline(dr_df: pd.DataFrame) -> pd.DataFrame:
    """
    Executes the IPTW + ANCOVA pipeline.
    1. Fits Logistic Regression for Propensity Score e(x).
    2. Calculates Weights W_i = e(x)/(1-e(x)) for controls, trimmed at 99th percentile.
    3. Fits WLS Regression to extract unbiased causal lift.
    """
    log.info("Running Doubly Robust Estimation (IPTW + ANCOVA) across all campaigns...")
    results = []
    
    for camp_id, group in dr_df.groupby("CAMPAIGN"):
        if group["TREATED"].nunique() < 2:
            continue
            
        try:
            # 1. Propensity Score Model
            # Add a tiny constant to PRE_SALES to help convergence if too many exact zeros exist
            group = group.copy()
            group["PRE_SALES_ADJ"] = group["PRE_SALES"] + 0.001 
            
            logit_model = smf.logit("TREATED ~ PRE_SALES_ADJ", data=group).fit(disp=False)
            group["PROPENSITY"] = logit_model.predict(group)
            
            # Bound propensity scores to prevent division by zero or infinite weights
            group["PROPENSITY"] = group["PROPENSITY"].clip(lower=0.001, upper=0.999)
            
            # 2. Assign IPTW Weights
            # Treated = 1.0, Control = e(x) / (1 - e(x))
            group["WEIGHT"] = np.where(
                group["TREATED"] == 1, 
                1.0, 
                group["PROPENSITY"] / (1.0 - group["PROPENSITY"])
            )
            
            # Trim extreme control weights at the 99th percentile to stabilize variance
            weight_cap = group.loc[group["TREATED"] == 0, "WEIGHT"].quantile(0.99)
            group.loc[group["TREATED"] == 0, "WEIGHT"] = group.loc[group["TREATED"] == 0, "WEIGHT"].clip(upper=weight_cap)
            
            # 3. Weighted Least Squares (ANCOVA)
            wls_model = smf.wls("CAMP_SALES ~ TREATED + PRE_SALES", data=group, weights=group["WEIGHT"]).fit()
            
            results.append({
                "CAMPAIGN": camp_id,
                "TRUE_CAUSAL_LIFT": wls_model.params["TREATED"],
                "LIFT_P_VALUE": wls_model.pvalues["TREATED"],
                "SIGNIFICANT_LIFT": bool(wls_model.pvalues["TREATED"] < 0.05),
                "BASELINE_COEF": wls_model.params["PRE_SALES"]
            })
            
        except Exception as e:
            log.warning(f"Campaign {camp_id} failed during Doubly Robust estimation: {e}")
            
    df_results = pd.DataFrame(results)
    log.info(f"Causal estimation complete for {len(df_results)} campaigns.")
    return df_results


# --------------------------------------------------------------------------- #
# Execution Pipeline Call
# --------------------------------------------------------------------------- #
# 1. Build the balanced dataset
dr_dataset = build_doubly_robust_dataset(cfg, tables["campaign_desc"], tables, pre_days=28)

# 2. Estimate True Causal Lift
causal_results = execute_doubly_robust_pipeline(dr_dataset)

# 3. Merge with Margin Economics to view the final causal ROI landscape
final_causal_summary = margin_econ[[
    "CAMPAIGN", "DESCRIPTION", "DISCOUNT_DEPTH_D", "UNPROFITABLE_BY_DESIGN"
]].merge(causal_results, on="CAMPAIGN", how="inner")

# Display the 10 campaigns sorted by their actual true incremental lift
display(final_causal_summary.sort_values("TRUE_CAUSAL_LIFT", ascending=False).head(10))

2026-08-07 10:12:03,168 | INFO | Constructing Pre-Campaign Baseline and Outcome matrix for all households...
2026-08-07 10:12:04,844 | INFO | Campaign 1: excluded 537 contaminated households from control pool
2026-08-07 10:12:04,846 | INFO | Campaign 30: excluded 99 contaminated households from control pool
2026-08-07 10:12:04,846 | INFO | Campaign 29: excluded 325 contaminated households from control pool
2026-08-07 10:12:04,847 | INFO | Campaign 28: excluded 405 contaminated households from control pool
2026-08-07 10:12:04,848 | INFO | Campaign 27: excluded 410 contaminated households from control pool
2026-08-07 10:12:04,849 | INFO | Campaign 26: excluded 3 contaminated households from control pool
2026-08-07 10:12:04,850 | INFO | Campaign 15: excluded 1379 contaminated households from control pool
2026-08-07 10:12:04,851 | INFO | Campaign 18: excluded 116 contaminated households from control pool
2026-08-07 10:12:04,851 | INFO | Campaign 17: excluded 1007 contaminated households fr

,CAMPAIGN,DESCRIPTION,DISCOUNT_DEPTH_D,UNPROFITABLE_BY_DESIGN,TRUE_CAUSAL_LIFT,LIFT_P_VALUE,SIGNIFICANT_LIFT,BASELINE_COEF
7,18,TypeA,1.148230,True,64.050677,3.597945e-22,True,1.293451
12,13,TypeA,1.265135,True,41.926865,2.293791e-17,True,1.096006
17,8,TypeA,1.074211,True,10.307667,1.241521e-02,True,1.111895
1,15,TypeC,0.811757,True,9.873782,4.285030e-03,True,4.513728
0,24,TypeB,1.315216,True,9.657217,7.841190e-20,True,0.691441
8,19,TypeB,1.549335,True,8.417573,3.668524e-11,True,0.523069
2,25,TypeB,1.089864,True,5.292785,2.899582e-19,True,0.549403
4,23,TypeB,1.004728,True,4.766634,1.717210e-11,True,0.815668
6,22,TypeB,0.717177,True,4.435794,2.111791e-19,True,0.445788
27,28,TypeB,0.505999,True,4.226935,8.779091e-17,True,0.445479


In [204]:
# --------------------------------------------------------------------------- #
# Incremental ROI: real discount cost, not just gross lift x margin
# --------------------------------------------------------------------------- #
# FIX vs. previous version: the old build_incremental_roi_table computed
# SHRUNK_ESTIMATE_PER_WEEK * margin and called that "incremental profit" --
# but that's gross sales lift x margin rate, with NO discount cost subtracted,
# despite the docstring citing doc 3's formula Delta_Pi = I*m - d*Q - F.
# DISCOUNT_DEPTH_PROXY was merged in as a column but never used in the math.
#
# This version:
#   1. Uses the REAL discount depth (DISCOUNT_DEPTH_D = p0 - p_paid, from base
#      price recovery in cell 6), not the SKU-count breadth proxy.
#   2. Computes incremental UNITS lift via a second Plan 1 DiD pass on
#      Y_ELIGIBLE_UNITS -- discount cost is d * units, not d * dollars, so the
#      sales-based lift alone can't price the discount side of the equation.
#   3. Actually subtracts d * incremental_units from the profit calc.
#
# CAVEAT CARRIED FORWARD (unchanged): naive placebo test still shows 18/30
# campaigns with significant pre-period differences. Both the sales-lift and
# units-lift DiD estimates below inherit that same confounding risk --
# PLACEBO_RELIABLE flags which campaigns' underlying counterfactual doesn't
# hold up even before touching the ROI math.

def build_incremental_roi_table(
    episodes,
    universal_outcomes,
    pooled_results,          # from Y_ELIGIBLE_SALES DiD (already computed)
    margin_econ,             # from compute_campaign_margin_economics (cell 6)
    hh_demographic,
    margin_scenarios=(0.20, 0.30, 0.40),
):
    """
    Combines incremental sales lift AND incremental units lift with real
    discount depth to produce true incremental profit:

        INCREMENTAL_PROFIT_PER_WEEK = sales_lift*m - units_lift*d

    where sales_lift/units_lift are both per-campaign-week DiD estimates
    (SHRUNK, i.e. empirical-Bayes pooled), m is the margin scenario, and d is
    DISCOUNT_DEPTH_D from margin_econ (p0 - p_paid, real, not a proxy).

    margin_econ's own standard_gross_margin=0.30 default is intentionally
    NOT the only margin used here -- margin_scenarios sweeps 0.20/0.30/0.40
    independently, since m is an assumption (no real margin data exists) and
    reporting a single point estimate overstates confidence, per the
    original review's smaller item on this.
    """
    log.info("Running second Plan 1 DiD pass on Y_ELIGIBLE_UNITS for incremental units lift...")
    plan1_units = run_plan1_event_study_did(
        episodes, universal_outcomes, hh_demographic, outcome_col="Y_ELIGIBLE_UNITS"
    )
    pooled_units = pool_campaign_effects(plan1_units, episodes)

    roi = pooled_results[["CAMPAIGN", "SHRUNK_ESTIMATE_PER_WEEK", "EMPIRICAL_P_VALUE",
                           "N_TREATED", "N_CONTROL", "IS_DIRECT_ESTIMATE"]].copy()
    roi = roi.rename(columns={"SHRUNK_ESTIMATE_PER_WEEK": "SALES_LIFT_PER_WEEK"})

    roi = roi.merge(
        pooled_units[["CAMPAIGN", "SHRUNK_ESTIMATE_PER_WEEK"]].rename(
            columns={"SHRUNK_ESTIMATE_PER_WEEK": "UNITS_LIFT_PER_WEEK"}
        ),
        on="CAMPAIGN", how="left",
    )

    roi = roi.merge(
        margin_econ[["CAMPAIGN", "DESCRIPTION", "BASE_PRICE_P0", "P_PAID",
                     "DISCOUNT_DEPTH_D", "BREAK_EVEN_KAPPA_STAR", "UNPROFITABLE_BY_DESIGN"]],
        on="CAMPAIGN", how="left",
    )

    n_missing_d = roi["DISCOUNT_DEPTH_D"].isna().sum()
    if n_missing_d:
        log.warning(f"{n_missing_d} campaigns missing DISCOUNT_DEPTH_D after merge -- "
                     f"their profit figures below will be NaN, not silently treated as zero discount cost.")
    roi["UNITS_LIFT_FOR_COST"] = roi["UNITS_LIFT_PER_WEEK"].clip(lower=0)

    for m in margin_scenarios:
        col = f"INCREMENTAL_PROFIT_PER_WEEK_m{int(m*100)}"
        roi[col] = (
        roi["SALES_LIFT_PER_WEEK"] * m
        - roi["UNITS_LIFT_FOR_COST"] * roi["DISCOUNT_DEPTH_D"])

    roi["NEGATIVE_LIFT_FLAG"] = roi["UNITS_LIFT_PER_WEEK"] < 0

    roi["PLACEBO_RELIABLE"] = roi["EMPIRICAL_P_VALUE"] >= 0.05
    n_unreliable = (~roi["PLACEBO_RELIABLE"]).sum()
    log.warning(f"{n_unreliable}/{len(roi)} campaigns fail the placebo test -- their ROI figures "
                f"below (sales lift, units lift, AND profit) should be read as unreliable, "
                f"not as confirmed numbers, regardless of sign.")
    mid_margin = margin_scenarios[len(margin_scenarios) // 2]
    sort_col = f"INCREMENTAL_PROFIT_PER_WEEK_m{int(mid_margin*100)}"

    return roi.sort_values(sort_col, ascending=False)


roi_table = build_incremental_roi_table(
    episodes, universal_outcomes, pooled_results, margin_econ, tables["hh_demographic"]
)

roi_table[[
    "CAMPAIGN", "DESCRIPTION", "SALES_LIFT_PER_WEEK", "UNITS_LIFT_PER_WEEK",
    "DISCOUNT_DEPTH_D", "EMPIRICAL_P_VALUE", "PLACEBO_RELIABLE",
    "INCREMENTAL_PROFIT_PER_WEEK_m20", "INCREMENTAL_PROFIT_PER_WEEK_m30", "INCREMENTAL_PROFIT_PER_WEEK_m40",
]]


2026-08-07 10:12:05,141 | INFO | Running second Plan 1 DiD pass on Y_ELIGIBLE_UNITS for incremental units lift...
2026-08-07 10:12:05,141 | INFO | Running Plan 1: matched stacked event-study DiD
2026-08-07 10:12:05,538 | INFO | Campaign 27: too few treated/control households after exclusions, skipping direct estimate (needs pooling)
2026-08-07 10:12:05,542 | INFO | Campaign 3: too few treated/control households after exclusions, skipping direct estimate (needs pooling)
2026-08-07 10:12:05,598 | INFO | Pooling campaign effects (empirical-Bayes shrinkage by campaign type)
2026-08-07 10:12:05,602 | INFO | Pooling complete: 28 campaigns had a direct estimate, 2 were fully pooled from their type group or the grand mean.
2026-08-07 10:12:05,605 | WARNING | 18/30 campaigns fail the placebo test -- their ROI figures below (sales lift, units lift, AND profit) should be read as unreliable, not as confirmed numbers, regardless of sign.


,CAMPAIGN,DESCRIPTION,SALES_LIFT_PER_WEEK,UNITS_LIFT_PER_WEEK,DISCOUNT_DEPTH_D,EMPIRICAL_P_VALUE,PLACEBO_RELIABLE,INCREMENTAL_PROFIT_PER_WEEK_m20,INCREMENTAL_PROFIT_PER_WEEK_m30,INCREMENTAL_PROFIT_PER_WEEK_m40
6,6,TypeC,12.577606,12.820236,4.440892e-16,0.003000,False,2.515521,3.773282,5.031042
3,27,TypeC,17.097653,17.166051,1.346528e-01,0.000667,False,1.108073,2.817839,4.527604
24,10,TypeB,0.167488,-0.358650,1.294171e+00,0.911833,True,0.033498,0.050246,0.066995
21,9,TypeB,1.314794,0.813204,4.605656e-01,0.422000,True,-0.111575,0.019905,0.151384
23,25,TypeB,1.112747,0.320227,1.089864e+00,0.486833,True,-0.126454,-0.015180,0.096095
25,2,TypeB,-0.104808,-0.273835,1.580239e+00,0.944500,True,-0.020962,-0.031442,-0.041923
26,24,TypeB,-1.309690,-2.036638,1.315216e+00,0.423833,True,-0.261938,-0.392907,-0.523876
20,11,TypeB,2.451942,2.330748,5.473953e-01,0.180833,True,-0.785452,-0.540258,-0.295064
22,5,TypeB,1.118821,0.643737,1.549855e+00,0.486000,True,-0.773934,-0.662052,-0.550170
27,7,TypeB,-2.471513,-2.610732,8.050146e-01,0.179000,True,-0.494303,-0.741454,-0.988605


In [205]:
# --------------------------------------------------------------------------- #
# Priority 2 fix: pre-period baseline must exclude days under ANY campaign's
# window store-wide, not just the focal campaign's own pre-window slice.
# --------------------------------------------------------------------------- #
#
# build_universal_outcomes() (a few cells up) computes Y_PRE_SALES as every
# dollar a household spent in [pre_start, start), with no check on whether
# some OTHER campaign was running store-wide during those days. A household
# can be quietly promoted by campaign B while we're using that same window as
# campaign A's "clean" baseline -- that inflates Y_PRE_SALES for anyone
# shopping through an overlapping promotion, which biases every downstream
# DiD estimate TOWARD ZERO (an already-elevated baseline makes during - pre
# look smaller than the true lift).
#
# Fix follows the same pattern already used in estimate_repurchase_cycles
# (Step 4.5): build a store-wide promoted-day lookup ONCE from all 30 campaign
# windows, then only count a pre-window transaction day toward Y_PRE_SALES if
# it is NOT inside any campaign's [START_DAY, END_DAY]. During/post windows
# are left untouched -- those are supposed to capture promotional effects, a
# "clean during" makes no sense.

def build_promoted_day_lookup(campaign_desc: pd.DataFrame):
    """Boolean Series indexed by DAY, True if that day falls inside ANY
    campaign's window store-wide. Only covers the campaign season
    [day_min, day_max]; days outside that range are unpromoted by
    construction and don't need a lookup entry."""
    windows = campaign_desc[["START_DAY", "END_DAY"]].to_numpy()
    day_min, day_max = int(campaign_desc["START_DAY"].min()), int(campaign_desc["END_DAY"].max())
    all_days = np.arange(day_min, day_max + 1)
    is_promoted = np.array([
        np.any((windows[:, 0] <= d) & (d <= windows[:, 1])) for d in all_days
    ])
    return pd.Series(is_promoted, index=all_days), day_min, day_max


def build_universal_outcomes_v2(
    cfg,
    tables,
    post_windows_by_campaign: dict | None = None,
    pre_period_weeks: int = 4,
    default_post_weeks: int = 4,
) -> pd.DataFrame:
    """
    Same as build_universal_outcomes, EXCEPT Y_PRE_SALES only accumulates
    transactions on days that are unpromoted STORE-WIDE (outside every
    campaign's window, not just this campaign's own [pre_start, start)
    slice). During/post windows are unchanged from v1.
    """
    log.info("Constructing universal outcome matrix (v2: store-wide-clean pre-period baseline)...")

    campaign_desc = tables["campaign_desc"]
    coupon = tables["coupon"]
    product = tables["product"]

    coupon_dedup = coupon.drop_duplicates(subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"])
    product_commodity = product.set_index("PRODUCT_ID")["COMMODITY_DESC"]

    elig = coupon_dedup[["CAMPAIGN", "PRODUCT_ID"]].drop_duplicates()
    elig["COMMODITY_DESC"] = elig["PRODUCT_ID"].map(product_commodity)
    elig_map = elig.groupby("CAMPAIGN")["PRODUCT_ID"].apply(set).to_dict()
    elig_commodity_map = elig.groupby("CAMPAIGN")["COMMODITY_DESC"].apply(lambda s: set(s.dropna())).to_dict()

    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]].to_dict("index")

    if post_windows_by_campaign is None:
        post_windows_by_campaign = {}

    promoted_lookup, season_min, season_max = build_promoted_day_lookup(campaign_desc)

    outcomes = {}
    METRIC_KEYS = [
        "Y_ELIGIBLE_SALES", "Y_ELIGIBLE_UNITS", "Y_CATEGORY_SALES",
        "Y_RIVAL_SALES", "Y_PRE_SALES", "Y_POST_SALES",
    ]

    def _init(key):
        if key not in outcomes:
            outcomes[key] = {k: 0.0 for k in METRIC_KEYS}
        return outcomes[key]

    n_pre_rows_total = 0
    n_pre_rows_dropped_contaminated = 0

    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk = chunk[chunk["QUANTITY"] > 0].copy()
        chunk["COMMODITY_DESC"] = chunk["PRODUCT_ID"].map(product_commodity)

        for camp_id, bounds in windows.items():
            start, end = bounds["START_DAY"], bounds["END_DAY"]
            pre_start = start - pre_period_weeks * 7
            post_weeks = post_windows_by_campaign.get(camp_id, default_post_weeks)
            post_end = end + int(post_weeks) * 7

            sub = chunk[(chunk["DAY"] >= pre_start) & (chunk["DAY"] <= post_end)]
            if sub.empty:
                continue

            elig_skus = elig_map.get(camp_id, set())
            elig_commodities = elig_commodity_map.get(camp_id, set())

            # --- Pre-period baseline: store-wide-clean days only (THE FIX) ---
            pre = sub[(sub["DAY"] >= pre_start) & (sub["DAY"] < start)]
            if not pre.empty:
                n_pre_rows_total += len(pre)
                inside_season = pre["DAY"].between(season_min, season_max)
                is_contaminated = pd.Series(False, index=pre.index)
                if inside_season.any():
                    is_contaminated.loc[inside_season] = promoted_lookup.reindex(
                        pre.loc[inside_season, "DAY"]
                    ).to_numpy()
                pre_clean = pre[~is_contaminated]
                n_pre_rows_dropped_contaminated += len(pre) - len(pre_clean)
                for hh, val in pre_clean.groupby("household_key")["SALES_VALUE"].sum().items():
                    _init((hh, camp_id))["Y_PRE_SALES"] += val

            # --- Campaign window (during) -- unchanged from v1 ---
            during = sub[(sub["DAY"] >= start) & (sub["DAY"] <= end)]
            if not during.empty:
                is_elig = during["PRODUCT_ID"].isin(elig_skus)
                same_commodity = during["COMMODITY_DESC"].isin(elig_commodities)

                elig_df = during[is_elig]
                for hh, g in elig_df.groupby("household_key"):
                    m = _init((hh, camp_id))
                    m["Y_ELIGIBLE_SALES"] += g["SALES_VALUE"].sum()
                    m["Y_ELIGIBLE_UNITS"] += g["QUANTITY"].sum()

                category_df = during[same_commodity]
                for hh, g in category_df.groupby("household_key"):
                    _init((hh, camp_id))["Y_CATEGORY_SALES"] += g["SALES_VALUE"].sum()

                rival_df = during[same_commodity & ~is_elig]
                for hh, g in rival_df.groupby("household_key"):
                    _init((hh, camp_id))["Y_RIVAL_SALES"] += g["SALES_VALUE"].sum()

            # --- Post-campaign window -- unchanged from v1 (captures
            # pull-forward/payback on purpose, not meant to be "clean") ---
            post = sub[(sub["DAY"] > end) & (sub["DAY"] <= post_end)]
            if not post.empty:
                for hh, val in post.groupby("household_key")["SALES_VALUE"].sum().items():
                    _init((hh, camp_id))["Y_POST_SALES"] += val

    if n_pre_rows_total:
        log.info(
            f"Pre-period baseline contamination: {n_pre_rows_dropped_contaminated}/{n_pre_rows_total} "
            f"({n_pre_rows_dropped_contaminated / n_pre_rows_total:.1%}) pre-window transaction rows fell "
            f"inside SOME campaign's active window store-wide and were excluded from Y_PRE_SALES."
        )

    records = [
        {"household_key": hh, "CAMPAIGN": camp_id, **metrics}
        for (hh, camp_id), metrics in outcomes.items()
    ]
    df_outcomes = pd.DataFrame(records)
    log.info(f"Universal outcomes table (v2) created with {len(df_outcomes):,} household-campaign records.")
    return df_outcomes


# --------------------------------------------------------------------------- #
# Re-run everything downstream with the clean baseline
# --------------------------------------------------------------------------- #
universal_outcomes_v2 = build_universal_outcomes_v2(
    cfg, tables, post_windows_by_campaign=campaign_post_windows_v3
)
universal_outcomes_v2 = universal_outcomes_v2.rename(columns={"household_key": "HOUSEHOLD_KEY"})
universal_outcomes_v2.to_csv("pipeline_output/universal_outcomes_v2_clean_baseline.csv", index=False)

plan1_results_v2 = run_plan1_event_study_did(episodes, universal_outcomes_v2, tables["hh_demographic"])
pooled_results_v2 = pool_campaign_effects(plan1_results_v2, episodes)
placebo_draws_naive_v2, pooled_results_v2 = run_placebo_test(episodes, universal_outcomes_v2, pooled_results_v2)

n_naive_failures_v2 = int((pooled_results_v2["EMPIRICAL_P_VALUE"] < 0.05).sum())
log.info(
    f"[v2, clean baseline] Naive placebo test: {n_naive_failures_v2}/30 campaigns still show "
    f"significant pre-period differences (v1 was 18/30 -- compare against this)"
)

pooled_results_v2[[
    "CAMPAIGN", "N_TREATED", "N_CONTROL", "DID_ESTIMATE_PER_WEEK",
    "SHRUNK_ESTIMATE_PER_WEEK", "EMPIRICAL_P_VALUE",
]].head(10)


2026-08-07 10:12:05,623 | INFO | Constructing universal outcome matrix (v2: store-wide-clean pre-period baseline)...
2026-08-07 10:12:10,968 | INFO | Pre-period baseline contamination: 3186105/3345134 (95.2%) pre-window transaction rows fell inside SOME campaign's active window store-wide and were excluded from Y_PRE_SALES.
2026-08-07 10:12:11,041 | INFO | Universal outcomes table (v2) created with 66,151 household-campaign records.
2026-08-07 10:12:11,213 | INFO | Running Plan 1: matched stacked event-study DiD
2026-08-07 10:12:11,667 | INFO | Campaign 27: too few treated/control households after exclusions, skipping direct estimate (needs pooling)
2026-08-07 10:12:11,671 | INFO | Campaign 3: too few treated/control households after exclusions, skipping direct estimate (needs pooling)
2026-08-07 10:12:11,726 | INFO | Pooling campaign effects (empirical-Bayes shrinkage by campaign type)
2026-08-07 10:12:11,730 | INFO | Pooling complete: 28 campaigns had a direct estimate, 2 were fully 

,CAMPAIGN,N_TREATED,N_CONTROL,DID_ESTIMATE_PER_WEEK,SHRUNK_ESTIMATE_PER_WEEK,EMPIRICAL_P_VALUE
0,18,1013,1152,15.285627,15.149190,0.000000
1,13,982,1259,10.697531,10.597624,0.000000
2,8,994,1230,7.290470,7.259169,0.000167
3,26,323,2019,4.961146,4.821294,0.000667
4,24,89,1252,2.203304,1.592398,0.016000
5,25,156,1637,1.817362,1.477709,0.020667
6,23,156,1591,1.449509,1.167873,0.032500
7,30,349,1845,1.027250,1.029233,0.040167
8,19,109,1046,1.504056,0.927351,0.046500
9,22,232,1085,0.804077,0.757524,0.061667


In [206]:
# --------------------------------------------------------------------------- #
# Priority 1a: Post-selection inference via data-splitting
# --------------------------------------------------------------------------- #
#
# A single 50/50 split of the WHOLE household universe (not a separate split
# per campaign) applied consistently everywhere -- a household is entirely in
# the "ranking" half or entirely in the "estimation" half for every campaign
# it could appear in, as either treated or control. Splitting per-campaign
# instead would let the same household serve as its own check in a different
# role across campaigns, which defeats the point.
#
# Half A: rank/select "winning" campaigns using SHRUNK_ESTIMATE_PER_WEEK.
# Half B: re-estimate ONLY the winners' DiD, independently, on data that
# played no role in selecting them. The Half-B SHRUNK_ESTIMATE_PER_WEEK is
# the winner's-curse-corrected number to report -- not the Half-A number,
# and not the original full-sample Plan 1 number.

def split_household_universe(universal_outcomes: pd.DataFrame, seed: int = 7):
    rng = np.random.default_rng(seed)
    all_hh = universal_outcomes["HOUSEHOLD_KEY"].unique()
    shuffled = rng.permutation(all_hh)
    half = len(shuffled) // 2
    half_a, half_b = set(shuffled[:half]), set(shuffled[half:])
    log.info(
        f"Post-selection split: {len(half_a)} households in ranking half (A), "
        f"{len(half_b)} in estimation half (B)"
    )
    return half_a, half_b


def run_post_selection_pipeline(
    episodes, universal_outcomes, hh_demographic,
    outcome_col: str = "Y_ELIGIBLE_SALES", top_n: int = 10, seed: int = 7,
):
    half_a, half_b = split_household_universe(universal_outcomes, seed=seed)

    ep_a = episodes[episodes["HOUSEHOLD_KEY"].isin(half_a)]
    uo_a = universal_outcomes[universal_outcomes["HOUSEHOLD_KEY"].isin(half_a)]
    ep_b = episodes[episodes["HOUSEHOLD_KEY"].isin(half_b)]
    uo_b = universal_outcomes[universal_outcomes["HOUSEHOLD_KEY"].isin(half_b)]

    log.info("Post-selection: running Plan 1 + pooling on ranking half (A)...")
    plan1_a = run_plan1_event_study_did(ep_a, uo_a, hh_demographic, outcome_col=outcome_col)
    pooled_a = pool_campaign_effects(plan1_a, ep_a)

    winners = (
        pooled_a.sort_values("SHRUNK_ESTIMATE_PER_WEEK", ascending=False)
        .head(top_n)["CAMPAIGN"].tolist()
    )
    log.info(f"Top {top_n} campaigns selected on half A: {winners}")

    log.info("Post-selection: running Plan 1 + pooling on estimation half (B), winners only...")
    plan1_b = run_plan1_event_study_did(ep_b, uo_b, hh_demographic, outcome_col=outcome_col)
    pooled_b = pool_campaign_effects(plan1_b, ep_b)

    unbiased = pooled_b[pooled_b["CAMPAIGN"].isin(winners)].copy()
    unbiased = unbiased.rename(columns={
        "SHRUNK_ESTIMATE_PER_WEEK": "UNBIASED_SHRUNK_ESTIMATE_PER_WEEK",
    })

    # Side-by-side: the (biased, winner's-curse-inflated) ranking estimate
    # next to the (unbiased, held-out) estimate for the same campaigns.
    compare = pooled_a[pooled_a["CAMPAIGN"].isin(winners)][
        ["CAMPAIGN", "SHRUNK_ESTIMATE_PER_WEEK"]
    ].rename(columns={"SHRUNK_ESTIMATE_PER_WEEK": "RANKING_ESTIMATE_PER_WEEK_A"}).merge(
        unbiased[["CAMPAIGN", "UNBIASED_SHRUNK_ESTIMATE_PER_WEEK"]], on="CAMPAIGN"
    )
    compare["WINNERS_CURSE_GAP"] = (
        compare["RANKING_ESTIMATE_PER_WEEK_A"] - compare["UNBIASED_SHRUNK_ESTIMATE_PER_WEEK"]
    )
    log.info(
        f"Mean winner's-curse gap (ranking estimate minus held-out estimate): "
        f"{compare['WINNERS_CURSE_GAP'].mean():.3f} per week -- positive means the ranking "
        f"half systematically overstated the winners, as the winner's-curse theory predicts."
    )

    return compare.sort_values("UNBIASED_SHRUNK_ESTIMATE_PER_WEEK", ascending=False)


# Run on the v2 (clean-baseline) outcomes from the previous cell, not v1.
post_selection_summary = run_post_selection_pipeline(
    episodes, universal_outcomes_v2, tables["hh_demographic"], top_n=10, seed=7
)
post_selection_summary.to_csv("pipeline_output/post_selection_unbiased_roi.csv", index=False)
post_selection_summary


2026-08-07 10:12:13,904 | INFO | Post-selection split: 1245 households in ranking half (A), 1246 in estimation half (B)
2026-08-07 10:12:13,908 | INFO | Post-selection: running Plan 1 + pooling on ranking half (A)...
2026-08-07 10:12:13,909 | INFO | Running Plan 1: matched stacked event-study DiD
2026-08-07 10:12:14,016 | INFO | Campaign 1: too few treated/control households after exclusions, skipping direct estimate (needs pooling)
2026-08-07 10:12:14,196 | INFO | Campaign 27: too few treated/control households after exclusions, skipping direct estimate (needs pooling)
2026-08-07 10:12:14,199 | INFO | Campaign 3: too few treated/control households after exclusions, skipping direct estimate (needs pooling)
2026-08-07 10:12:14,235 | INFO | Campaign 15: too few treated/control households after exclusions, skipping direct estimate (needs pooling)
2026-08-07 10:12:14,236 | INFO | Pooling campaign effects (empirical-Bayes shrinkage by campaign type)
2026-08-07 10:12:14,240 | INFO | Pooling 

,CAMPAIGN,RANKING_ESTIMATE_PER_WEEK_A,UNBIASED_SHRUNK_ESTIMATE_PER_WEEK,WINNERS_CURSE_GAP
0,18,17.694490,17.371036,0.323453
1,13,13.943560,10.825041,3.118519
2,8,8.130227,4.634259,3.495968
3,26,2.161361,2.968803,-0.807442
4,24,1.925594,1.013996,0.911598
6,22,1.437048,0.735809,0.701239
5,25,1.571525,0.727951,0.843574
7,23,1.161281,0.722843,0.438437
9,9,0.788167,0.563555,0.224613
8,12,0.878461,-0.072387,0.950849


In [207]:
# --------------------------------------------------------------------------- #
# Priority 1b: Bootstrap ROI intervals + Fieller's theorem for the ROI ratio
# --------------------------------------------------------------------------- #
#
# 1. run_plan1_paired_matching() re-runs Plan 1's matching ONCE per campaign
#    (identical propensity model / nearest-neighbor logic to
#    run_plan1_event_study_did) but returns the raw per-pair arrays for BOTH
#    sales and units, matched from the SAME treated-control pairs. That's
#    what makes a joint bootstrap valid -- sales lift and units lift for a
#    given campaign are correlated (same households), and resampling them
#    independently would understate that correlation.
#
# 2. bootstrap_roi_intervals() resamples matched pairs WITH replacement
#    (same drawn indices applied to sales and units together) to build a
#    percentile CI for weekly incremental profit, then applies Fieller's
#    theorem to the RATIO (incremental gross profit) / (incremental discount
#    cost) using the bootstrap covariance of that numerator and denominator --
#    this is the right tool for a ratio of two noisy, correlated estimates
#    (a plain bootstrap on the ratio itself can misbehave when the
#    denominator gets close to zero; Fieller's approach is built for that).
#
# CARRIED-FORWARD QUIRK (not fixed here, flagging it explicitly): the
# existing pipeline's units-lift DiD subtracts Y_PRE_SALES (dollars) from
# Y_ELIGIBLE_UNITS (units) as its "pre" baseline, because no Y_PRE_UNITS
# column exists anywhere in the pipeline. This bootstrap replicates that same
# math on purpose, so the interval is around the SAME quantity roi_table
# already reports -- fixing the units/dollars mismatch is a separate task
# (would need a new Y_PRE_UNITS column threaded through build_universal_outcomes).

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


def run_plan1_paired_matching(episodes, universal_outcomes, demographics):
    """
    Same treated/control construction and matching as run_plan1_event_study_did,
    but returns per-campaign dicts of paired arrays instead of a single summary
    row, so downstream code can bootstrap-resample matched pairs directly.
    """
    log.info("Running Plan 1 matching (paired-array version, for bootstrap)...")
    linked_campaigns_by_hh = episodes.groupby("HOUSEHOLD_KEY")["CAMPAIGN"].apply(set).to_dict()
    concurrent_free_hh = set(episodes.loc[episodes["N_CONCURRENT_CAMPAIGNS"].eq(0), "HOUSEHOLD_KEY"])

    windows = episodes.drop_duplicates("CAMPAIGN").set_index("CAMPAIGN")[["START_DAY", "END_DAY", "DURATION_DAYS"]]
    all_campaign_windows = windows[["START_DAY", "END_DAY"]]

    demo_cols = [c for c in demographics.columns if c.endswith("_DESC")]
    uo = universal_outcomes.merge(demographics, on="HOUSEHOLD_KEY", how="left")

    paired = {}
    duration_weeks_by_campaign = {}

    for campaign in windows.index:
        c_start, c_end = windows.loc[campaign, ["START_DAY", "END_DAY"]]
        duration_weeks = windows.loc[campaign, "DURATION_DAYS"] / 7.0
        duration_weeks_by_campaign[campaign] = duration_weeks

        treated_hh = set(episodes.loc[episodes["CAMPAIGN"] == campaign, "HOUSEHOLD_KEY"]) & concurrent_free_hh

        def _overlaps_campaign(other_campaign):
            o_start, o_end = all_campaign_windows.loc[other_campaign]
            return c_start <= o_end and o_start <= c_end

        overlapping_campaigns = {c for c in all_campaign_windows.index if c != campaign and _overlaps_campaign(c)}

        control_hh = {
            hh for hh, camps in linked_campaigns_by_hh.items()
            if hh not in treated_hh and not (camps & ({campaign} | overlapping_campaigns))
        }
        control_hh |= set(universal_outcomes["HOUSEHOLD_KEY"].unique()) - set(linked_campaigns_by_hh.keys()) - treated_hh

        treated_rows = uo[(uo["CAMPAIGN"] == campaign) & (uo["HOUSEHOLD_KEY"].isin(treated_hh))].copy()
        control_rows = uo[(uo["CAMPAIGN"] == campaign) & (uo["HOUSEHOLD_KEY"].isin(control_hh))].copy()

        if len(treated_rows) < 10 or len(control_rows) < 10:
            continue

        combo = pd.concat([treated_rows.assign(_T=1), control_rows.assign(_T=0)], ignore_index=True)
        feature_cols = demo_cols + ["Y_PRE_SALES"]
        X = pd.get_dummies(combo[feature_cols], dummy_na=True).fillna(0)

        try:
            X_scaled = StandardScaler().fit_transform(X)
            ps_model = LogisticRegression(max_iter=2000)
            ps_model.fit(X_scaled, combo["_T"])
            combo["_pscore"] = ps_model.predict_proba(X_scaled)[:, 1]
        except Exception:
            combo["_pscore"] = combo["Y_PRE_SALES"]

        treated_idx = combo[combo["_T"] == 1].index
        control_idx = combo[combo["_T"] == 0].index
        nn = NearestNeighbors(n_neighbors=1).fit(combo.loc[control_idx, ["_pscore"]])
        _, match_pos = nn.kneighbors(combo.loc[treated_idx, ["_pscore"]])
        matched_control_idx = combo.loc[control_idx].iloc[match_pos.flatten()].index

        paired[campaign] = {
            "sales_treated": combo.loc[treated_idx, "Y_ELIGIBLE_SALES"].to_numpy(),
            "sales_control": combo.loc[matched_control_idx, "Y_ELIGIBLE_SALES"].to_numpy(),
            "units_treated": combo.loc[treated_idx, "Y_ELIGIBLE_UNITS"].to_numpy(),
            "units_control": combo.loc[matched_control_idx, "Y_ELIGIBLE_UNITS"].to_numpy(),
            "pre_treated": combo.loc[treated_idx, "Y_PRE_SALES"].to_numpy(),
            "pre_control": combo.loc[matched_control_idx, "Y_PRE_SALES"].to_numpy(),
        }

    return paired, duration_weeks_by_campaign


def bootstrap_roi_intervals(
    paired, duration_weeks_by_campaign, margin_econ,
    margin: float = 0.30, n_boot: int = 2000, seed: int = 42, ci: float = 0.95,
):
    """
    For each campaign with matched pairs:
      - Bootstrap-resample pair indices WITH replacement, n_boot times.
      - Each draw recomputes sales-DiD and units-DiD per week using the SAME
        resampled indices for both (preserves their correlation).
      - INCREMENTAL_PROFIT_PER_WEEK = sales_lift*margin - max(units_lift,0)*discount_depth_d
      - Percentile CI on profit from the bootstrap distribution.
      - Fieller's theorem CI on (sales_lift*margin) / (max(units_lift,0)*d),
        using the bootstrap mean/variance/covariance of numerator and
        denominator (valid without assuming the ratio itself is normal).
    """
    z = 1.959963985  # 95% two-sided normal critical value
    rng = np.random.default_rng(seed)
    d_lookup = margin_econ.set_index("CAMPAIGN")["DISCOUNT_DEPTH_D"]

    rows = []
    for campaign, arrs in paired.items():
        n_pairs = len(arrs["sales_treated"])
        duration_weeks = max(duration_weeks_by_campaign.get(campaign, 1.0), 1e-9)
        d = d_lookup.get(campaign, np.nan)
        if pd.isna(d) or n_pairs < 5:
            continue

        boot_profit = np.empty(n_boot)
        boot_numerator = np.empty(n_boot)    # Return = sales_lift * margin, per week
        boot_denominator = np.empty(n_boot)  # Investment = units_lift_for_cost * d, per week

        for b in range(n_boot):
            idx = rng.integers(0, n_pairs, size=n_pairs)  # resample matched PAIRS, with replacement
            sales_lift = np.mean(
                (arrs["sales_treated"][idx] - arrs["pre_treated"][idx])
                - (arrs["sales_control"][idx] - arrs["pre_control"][idx])
            ) / duration_weeks
            # NOTE: mirrors the existing pipeline's units-lift math (subtracts
            # Y_PRE_SALES dollars as the "pre" baseline for a units outcome --
            # see the module docstring above). Not fixed here on purpose.
            units_lift = np.mean(
                (arrs["units_treated"][idx] - arrs["pre_treated"][idx])
                - (arrs["units_control"][idx] - arrs["pre_control"][idx])
            ) / duration_weeks
            units_lift_for_cost = max(units_lift, 0.0)

            numerator = sales_lift * margin
            denominator = units_lift_for_cost * d
            boot_numerator[b] = numerator
            boot_denominator[b] = denominator
            boot_profit[b] = numerator - denominator

        profit_lo, profit_hi = np.percentile(boot_profit, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])

        R_bar, D_bar = boot_numerator.mean(), boot_denominator.mean()
        s_R2, s_D2 = boot_numerator.var(ddof=1), boot_denominator.var(ddof=1)
        s_RD = np.cov(boot_numerator, boot_denominator, ddof=1)[0, 1]

        fieller_note = "ok"
        theta_lo = theta_hi = np.nan
        if D_bar != 0:
            theta_hat = R_bar / D_bar
            g = (z ** 2 * s_D2) / (D_bar ** 2)
            if g < 1:
                center = theta_hat - g * (s_RD / s_D2) if s_D2 > 0 else theta_hat
                inside = (
                    s_R2 - 2 * theta_hat * s_RD + theta_hat ** 2 * s_D2
                    - g * (s_R2 - (s_RD ** 2 / s_D2 if s_D2 > 0 else 0.0))
                )
                if inside >= 0:
                    spread = (z / D_bar) * np.sqrt(inside)
                    theta_lo, theta_hi = (center - spread) / (1 - g), (center + spread) / (1 - g)
                else:
                    fieller_note = "negative discriminant -- CI undefined"
            else:
                fieller_note = "denominator not significantly different from 0 (g >= 1) -- Fieller CI unbounded"
        else:
            fieller_note = "denominator is exactly 0 (no discount cost) -- ratio undefined"

        rows.append({
            "CAMPAIGN": campaign,
            "N_PAIRS": n_pairs,
            "PROFIT_PER_WEEK_POINT": boot_profit.mean(),
            "PROFIT_CI_LOW": profit_lo,
            "PROFIT_CI_HIGH": profit_hi,
            "ROI_RATIO_POINT": R_bar / D_bar if D_bar != 0 else np.nan,
            "ROI_FIELLER_CI_LOW": theta_lo,
            "ROI_FIELLER_CI_HIGH": theta_hi,
            "FIELLER_NOTE": fieller_note,
        })

    out = pd.DataFrame(rows).sort_values("PROFIT_PER_WEEK_POINT", ascending=False)
    log.info(
        f"Bootstrap + Fieller ROI intervals computed for {len(out)} campaigns "
        f"({n_boot} resamples each, margin={margin:.0%})."
    )
    return out


paired_arrays, duration_weeks_by_campaign = run_plan1_paired_matching(
    episodes, universal_outcomes_v2, tables["hh_demographic"]
)
roi_intervals = bootstrap_roi_intervals(
    paired_arrays, duration_weeks_by_campaign, margin_econ, margin=0.30, n_boot=2000
)
roi_intervals.to_csv("pipeline_output/roi_bootstrap_fieller_intervals.csv", index=False)
roi_intervals.head(15)


2026-08-07 10:12:14,572 | INFO | Running Plan 1 matching (paired-array version, for bootstrap)...
2026-08-07 10:12:15,656 | INFO | Bootstrap + Fieller ROI intervals computed for 28 campaigns (2000 resamples each, margin=30%).


,CAMPAIGN,N_PAIRS,PROFIT_PER_WEEK_POINT,PROFIT_CI_LOW,PROFIT_CI_HIGH,ROI_RATIO_POINT,ROI_FIELLER_CI_LOW,ROI_FIELLER_CI_HIGH,FIELLER_NOTE
10,5,144,0.075474,-0.411398,0.315452,1.418747e+00,NaN,NaN,denominator not significantly different from 0...
26,6,47,0.009185,0.000000,0.027905,1.546986e+15,NaN,NaN,denominator not significantly different from 0...
19,21,58,0.007699,-0.071395,0.089962,2.282495e+01,NaN,NaN,denominator not significantly different from 0...
9,4,70,0.005591,-0.032359,0.043482,1.183984e+00,NaN,NaN,denominator not significantly different from 0...
12,9,151,0.003289,-0.055328,0.063397,1.032119e+00,-0.465027,1.422170,ok
25,20,190,-0.019811,-0.035992,-0.003274,8.290833e-01,0.669007,0.973993,ok
5,28,15,-0.021470,-0.246170,0.115350,5.409691e-01,NaN,NaN,denominator not significantly different from 0...
24,14,170,-0.022565,-0.050565,0.003317,5.472562e-01,-1.052265,1.071121,ok
11,7,152,-0.024914,-0.048243,0.000800,-4.704131e+04,NaN,NaN,denominator not significantly different from 0...
18,19,109,-0.061263,-0.721908,0.458473,8.807563e-01,NaN,NaN,denominator not significantly different from 0...


In [211]:
# --------------------------------------------------------------------------- #
# Unify everything into one master leaderboard, one row per campaign
# --------------------------------------------------------------------------- #
#
# Up to this point, sales-lift (pooled_results_v2), profit/ROI intervals
# (roi_intervals), and winner's-curse correction (post_selection_summary)
# live in three separate tables, each truncated by a head() call for display.
# This merges the FULL versions of all three (plus margin_econ's break-even
# flag) into one leaderboard with an explicit VERDICT per campaign, so a
# strong sales lift that doesn't survive the profit/placebo/post-selection
# checks doesn't get reported as a win just because it was the top row in
# one of the earlier tables.

def build_master_leaderboard(pooled_results_v2, roi_intervals, post_selection_summary, margin_econ):
    df = pooled_results_v2[[
        "CAMPAIGN", "N_TREATED", "N_CONTROL", "SHRUNK_ESTIMATE_PER_WEEK", "EMPIRICAL_P_VALUE"
    ]].copy()
    df = df.rename(columns={"SHRUNK_ESTIMATE_PER_WEEK": "SALES_LIFT_PER_WEEK_CLEAN_BASELINE"})
    df["PLACEBO_RELIABLE"] = df["EMPIRICAL_P_VALUE"] >= 0.05

    df = df.merge(
        roi_intervals[[
            "CAMPAIGN", "N_PAIRS", "PROFIT_PER_WEEK_POINT", "PROFIT_CI_LOW", "PROFIT_CI_HIGH",
            "ROI_RATIO_POINT", "ROI_FIELLER_CI_LOW", "ROI_FIELLER_CI_HIGH", "FIELLER_NOTE",
        ]],
        on="CAMPAIGN", how="left",
    )

    df["PROFIT_SIGN"] = np.where(
        df["PROFIT_CI_LOW"] > 0, "POSITIVE",
        np.where(df["PROFIT_CI_HIGH"] < 0, "NEGATIVE", "INCONCLUSIVE"),
    )

    post_sel = post_selection_summary[[
        "CAMPAIGN", "UNBIASED_SHRUNK_ESTIMATE_PER_WEEK", "WINNERS_CURSE_GAP",
    ]].rename(columns={"UNBIASED_SHRUNK_ESTIMATE_PER_WEEK": "POST_SELECTION_UNBIASED_SALES_LIFT"})
    df = df.merge(post_sel, on="CAMPAIGN", how="left")
    df["WAS_POST_SELECTION_WINNER"] = df["CAMPAIGN"].isin(post_selection_summary["CAMPAIGN"])

    df = df.merge(margin_econ[["CAMPAIGN", "UNPROFITABLE_BY_DESIGN"]], on="CAMPAIGN", how="left")

    def _verdict(row):
        if row["WAS_POST_SELECTION_WINNER"] and pd.notna(row["POST_SELECTION_UNBIASED_SALES_LIFT"]) \
                and row["POST_SELECTION_UNBIASED_SALES_LIFT"] <= 0:
            return "FALSE POSITIVE -- winner's curse (fails on held-out half)"
        if pd.notna(row["PLACEBO_RELIABLE"]) and not row["PLACEBO_RELIABLE"]:
            return "UNRELIABLE -- fails placebo test"
        if row["PROFIT_SIGN"] == "NEGATIVE":
            return "CONFIRMED LOSS"
        if row["PROFIT_SIGN"] == "POSITIVE":
            return "CONFIRMED PROFIT"
        if pd.isna(row["PROFIT_PER_WEEK_POINT"]):
            return "NOT BOOTSTRAPPED -- too few matched pairs or missing discount depth"
        return "INCONCLUSIVE -- CI too wide"

    df["VERDICT"] = df.apply(_verdict, axis=1)
    return df.sort_values("PROFIT_PER_WEEK_POINT", ascending=False, na_position="last")


master_leaderboard = build_master_leaderboard(
    pooled_results_v2, roi_intervals, post_selection_summary, margin_econ
)
master_leaderboard.to_csv("pipeline_output/master_campaign_leaderboard.csv", index=False)

log.info("Verdict counts across all campaigns:\n" + master_leaderboard["VERDICT"].value_counts().to_string())

master_leaderboard


2026-08-07 10:22:12,606 | INFO | Verdict counts across all campaigns:
VERDICT
CONFIRMED LOSS                                                         10
UNRELIABLE -- fails placebo test                                        9
INCONCLUSIVE -- CI too wide                                             8
NOT BOOTSTRAPPED -- too few matched pairs or missing discount depth     2
FALSE POSITIVE -- winner's curse (fails on held-out half)               1


,CAMPAIGN,N_TREATED,N_CONTROL,SALES_LIFT_PER_WEEK_CLEAN_BASELINE,EMPIRICAL_P_VALUE,PLACEBO_RELIABLE,N_PAIRS,PROFIT_PER_WEEK_POINT,PROFIT_CI_LOW,PROFIT_CI_HIGH,ROI_RATIO_POINT,ROI_FIELLER_CI_LOW,ROI_FIELLER_CI_HIGH,FIELLER_NOTE,PROFIT_SIGN,POST_SELECTION_UNBIASED_SALES_LIFT,WINNERS_CURSE_GAP,WAS_POST_SELECTION_WINNER,UNPROFITABLE_BY_DESIGN,VERDICT
10,5,144,1853,0.682373,0.070500,True,144.0,0.075474,-0.411398,0.315452,1.418747e+00,NaN,NaN,denominator not significantly different from 0...,INCONCLUSIVE,NaN,NaN,False,True,INCONCLUSIVE -- CI too wide
23,6,47,1005,0.033203,0.558000,True,47.0,0.009185,0.000000,0.027905,1.546986e+15,NaN,NaN,denominator not significantly different from 0...,INCONCLUSIVE,NaN,NaN,False,False,INCONCLUSIVE -- CI too wide
24,21,58,1085,0.028009,0.607000,True,58.0,0.007699,-0.071395,0.089962,2.282495e+01,NaN,NaN,denominator not significantly different from 0...,INCONCLUSIVE,NaN,NaN,False,True,INCONCLUSIVE -- CI too wide
18,4,70,1799,0.116322,0.231167,True,70.0,0.005591,-0.032359,0.043482,1.183984e+00,NaN,NaN,denominator not significantly different from 0...,INCONCLUSIVE,NaN,NaN,False,True,INCONCLUSIVE -- CI too wide
13,9,151,1151,0.308258,0.130500,True,151.0,0.003289,-0.055328,0.063397,1.032119e+00,-0.465027,1.422170,ok,INCONCLUSIVE,0.563555,0.224613,True,True,INCONCLUSIVE -- CI too wide
14,20,190,994,0.289005,0.134833,True,190.0,-0.019811,-0.035992,-0.003274,8.290833e-01,0.669007,0.973993,ok,NEGATIVE,NaN,NaN,False,True,CONFIRMED LOSS
22,28,15,1861,0.070240,0.344167,True,15.0,-0.021470,-0.246170,0.115350,5.409691e-01,NaN,NaN,denominator not significantly different from 0...,INCONCLUSIVE,NaN,NaN,False,True,INCONCLUSIVE -- CI too wide
19,14,170,962,0.092714,0.277167,True,170.0,-0.022565,-0.050565,0.003317,5.472562e-01,-1.052265,1.071121,ok,INCONCLUSIVE,NaN,NaN,False,True,INCONCLUSIVE -- CI too wide
25,7,152,1141,-0.082962,0.309500,True,152.0,-0.024914,-0.048243,0.000800,-4.704131e+04,NaN,NaN,denominator not significantly different from 0...,INCONCLUSIVE,NaN,NaN,False,True,INCONCLUSIVE -- CI too wide
8,19,109,1046,0.927351,0.046500,False,109.0,-0.061263,-0.721908,0.458473,8.807563e-01,NaN,NaN,denominator not significantly different from 0...,INCONCLUSIVE,NaN,NaN,False,True,UNRELIABLE -- fails placebo test


In [212]:
# --------------------------------------------------------------------------- #
# Priority 4: Penalized-regression + Coarsened Exact Matching (CEM) fallback
# for small-sample campaigns (currently: 3 and 27, N_TREATED = 7 and 9 --
# both routed straight to full type-mean pooling, so neither has ANY real
# information in the leaderboard right now; both show IDENTICAL
# SALES_LIFT_PER_WEEK_CLEAN_BASELINE = 0.088967 because they inherited the
# same group mean).
# --------------------------------------------------------------------------- #
#
# Two fixes, tried in order per campaign:
#   1. Penalized (small-C) logistic regression propensity matching -- same
#      structure as Plan 1's matching, but C=0.1 instead of the default 1.0.
#      With ~7-9 demographic dummies and only 7-9 treated rows, standard
#      logistic regression is prone to quasi-separation even when sklearn's
#      solver doesn't literally throw -- it "succeeds" but produces
#      degenerate near-0/near-1 propensity scores. Strong L2 shrinkage
#      keeps that from happening.
#   2. If even penalized logistic regression yields degenerate scores
#      (checked directly: are >90% of scores pinned to the extremes?), fall
#      back to Coarsened Exact Matching: bucket households into strata by
#      demographic combination, match only within an exact stratum, backing
#      off to a coarser stratum if a treated household's exact stratum has
#      no control at all.

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

CEM_VARS_FULL = ["AGE_DESC", "INCOME_DESC", "HH_COMP_DESC", "HOMEOWNER_DESC"]
CEM_VARS_COARSE = ["AGE_DESC", "INCOME_DESC"]  # backoff level


def _cem_stratum_key(df, cols):
    return df[cols].fillna("MISSING").astype(str).agg("|".join, axis=1)


def match_via_cem(treated_df, control_df):
    """
    Two-level backoff: exact match on CEM_VARS_FULL, then CEM_VARS_COARSE,
    then random match against the whole control pool as a last resort
    (logged explicitly, never silent). With-replacement, since small strata
    will often need to reuse the same control multiple times.
    """
    treated_df = treated_df.copy()
    control_df = control_df.copy()
    treated_df["_stratum_full"] = _cem_stratum_key(treated_df, CEM_VARS_FULL)
    control_df["_stratum_full"] = _cem_stratum_key(control_df, CEM_VARS_FULL)
    treated_df["_stratum_coarse"] = _cem_stratum_key(treated_df, CEM_VARS_COARSE)
    control_df["_stratum_coarse"] = _cem_stratum_key(control_df, CEM_VARS_COARSE)

    control_by_full = control_df.groupby("_stratum_full").groups
    control_by_coarse = control_df.groupby("_stratum_coarse").groups

    matched_control_idx = []
    n_full, n_coarse, n_random = 0, 0, 0
    rng = np.random.default_rng(0)

    for _, row in treated_df.iterrows():
        pool = control_by_full.get(row["_stratum_full"])
        if pool is not None and len(pool):
            matched_control_idx.append(rng.choice(pool)); n_full += 1; continue
        pool = control_by_coarse.get(row["_stratum_coarse"])
        if pool is not None and len(pool):
            matched_control_idx.append(rng.choice(pool)); n_coarse += 1; continue
        matched_control_idx.append(rng.choice(control_df.index.to_numpy())); n_random += 1

    log.info(
        f"CEM matching: {n_full} exact-stratum matches, {n_coarse} coarse-backoff "
        f"matches, {n_random} random-fallback matches (no comparable stratum at all)"
    )
    return treated_df.index.to_numpy(), np.array(matched_control_idx)


def run_plan1_small_sample_fallback(
    episodes, universal_outcomes, demographics,
    campaigns_to_fix, outcome_col: str = "Y_ELIGIBLE_SALES",
):
    """Re-estimates ONLY the campaigns in campaigns_to_fix, using penalized
    logistic regression first and CEM as a second-line fallback, instead of
    Plan 1's original behavior of skipping straight to full pooling."""
    log.info(f"Running small-sample fallback matching for campaigns: {campaigns_to_fix}")

    linked_campaigns_by_hh = episodes.groupby("HOUSEHOLD_KEY")["CAMPAIGN"].apply(set).to_dict()
    concurrent_free_hh = set(episodes.loc[episodes["N_CONCURRENT_CAMPAIGNS"].eq(0), "HOUSEHOLD_KEY"])
    windows = episodes.drop_duplicates("CAMPAIGN").set_index("CAMPAIGN")[["START_DAY", "END_DAY", "DURATION_DAYS"]]
    all_campaign_windows = windows[["START_DAY", "END_DAY"]]
    demo_cols = [c for c in demographics.columns if c.endswith("_DESC")]
    uo = universal_outcomes.merge(demographics, on="HOUSEHOLD_KEY", how="left")

    results = []
    for campaign in campaigns_to_fix:
        if campaign not in windows.index:
            continue
        c_start, c_end = windows.loc[campaign, ["START_DAY", "END_DAY"]]
        duration_weeks = windows.loc[campaign, "DURATION_DAYS"] / 7.0

        treated_hh = set(episodes.loc[episodes["CAMPAIGN"] == campaign, "HOUSEHOLD_KEY"]) & concurrent_free_hh

        def _overlaps_campaign(other_campaign):
            o_start, o_end = all_campaign_windows.loc[other_campaign]
            return c_start <= o_end and o_start <= c_end

        overlapping_campaigns = {c for c in all_campaign_windows.index if c != campaign and _overlaps_campaign(c)}
        control_hh = {
            hh for hh, camps in linked_campaigns_by_hh.items()
            if hh not in treated_hh and not (camps & ({campaign} | overlapping_campaigns))
        }
        control_hh |= set(universal_outcomes["HOUSEHOLD_KEY"].unique()) - set(linked_campaigns_by_hh.keys()) - treated_hh

        treated_rows = uo[(uo["CAMPAIGN"] == campaign) & (uo["HOUSEHOLD_KEY"].isin(treated_hh))].copy()
        control_rows = uo[(uo["CAMPAIGN"] == campaign) & (uo["HOUSEHOLD_KEY"].isin(control_hh))].copy()

        if len(treated_rows) < 2 or len(control_rows) < 2:
            log.warning(f"Campaign {campaign}: fewer than 2 treated/control rows even for fallback -- skipping")
            continue

        combo = pd.concat([treated_rows.assign(_T=1), control_rows.assign(_T=0)], ignore_index=True)
        feature_cols = demo_cols + ["Y_PRE_SALES"]
        X = pd.get_dummies(combo[feature_cols], dummy_na=True).fillna(0)

        used_method = "penalized_logit"
        try:
            X_scaled = StandardScaler().fit_transform(X)
            ps_model = LogisticRegression(C=0.1, max_iter=2000)  # small C = strong L2 shrinkage
            ps_model.fit(X_scaled, combo["_T"])
            scores = ps_model.predict_proba(X_scaled)[:, 1]
            degenerate_share = np.mean((scores < 0.01) | (scores > 0.99))
            if degenerate_share > 0.9:
                raise ValueError(f"degenerate propensity scores ({degenerate_share:.0%} at the extremes)")
            combo["_pscore"] = scores
            treated_idx = combo[combo["_T"] == 1].index
            control_idx = combo[combo["_T"] == 0].index
            nn = NearestNeighbors(n_neighbors=1).fit(combo.loc[control_idx, ["_pscore"]])
            _, match_pos = nn.kneighbors(combo.loc[treated_idx, ["_pscore"]])
            matched_control_idx = combo.loc[control_idx].iloc[match_pos.flatten()].index
        except Exception as e:
            used_method = "cem"
            log.info(f"Campaign {campaign}: penalized logit unusable ({e}) -- falling back to CEM")
            treated_idx, matched_control_idx = match_via_cem(
                combo[combo["_T"] == 1], combo[combo["_T"] == 0]
            )

        treated_during = combo.loc[treated_idx, outcome_col].to_numpy()
        treated_pre = combo.loc[treated_idx, "Y_PRE_SALES"].to_numpy()
        control_during = combo.loc[matched_control_idx, outcome_col].to_numpy()
        control_pre = combo.loc[matched_control_idx, "Y_PRE_SALES"].to_numpy()

        did_per_pair = (treated_during - treated_pre) - (control_during - control_pre)
        did_estimate = np.nanmean(did_per_pair) / max(duration_weeks, 1e-9)
        se = np.nanstd(did_per_pair, ddof=1) / np.sqrt(len(did_per_pair)) / max(duration_weeks, 1e-9)

        results.append({
            "CAMPAIGN": campaign, "N_TREATED": len(treated_rows), "N_CONTROL": len(control_rows),
            "DID_ESTIMATE_PER_WEEK": did_estimate, "SE_PER_WEEK": se, "MATCH_METHOD": used_method,
            "NOTE": f"small-sample fallback ({used_method}) -- direct estimate now available",
        })

    return pd.DataFrame(results)


# --------------------------------------------------------------------------- #
# Apply: replace campaigns 3 and 27's fully-pooled placeholder with a real
# direct estimate, then re-pool everything so SHRUNK_ESTIMATE reflects
# actual matched data instead of the type-group mean.
# --------------------------------------------------------------------------- #
small_sample_campaigns = pooled_results_v2.loc[
    pooled_results_v2["N_TREATED"] < 10, "CAMPAIGN"
].tolist()  # currently [3, 27]

fallback_results = run_plan1_small_sample_fallback(
    episodes, universal_outcomes_v2, tables["hh_demographic"], small_sample_campaigns
)

plan1_results_v3 = plan1_results_v2.copy()
for _, row in fallback_results.iterrows():
    mask = plan1_results_v3["CAMPAIGN"] == row["CAMPAIGN"]
    for col in ["N_TREATED", "N_CONTROL", "DID_ESTIMATE_PER_WEEK", "SE_PER_WEEK", "NOTE"]:
        plan1_results_v3.loc[mask, col] = row[col]

pooled_results_v3 = pool_campaign_effects(plan1_results_v3, episodes)
pooled_results_v3[pooled_results_v3["CAMPAIGN"].isin(small_sample_campaigns)][[
    "CAMPAIGN", "N_TREATED", "N_CONTROL", "DID_ESTIMATE_PER_WEEK",
    "SHRUNK_ESTIMATE_PER_WEEK", "IS_DIRECT_ESTIMATE",
]]

2026-08-07 10:27:42,651 | INFO | Running small-sample fallback matching for campaigns: [27, 3]
2026-08-07 10:27:42,712 | INFO | Campaign 27: penalized logit unusable (degenerate propensity scores (94% at the extremes)) -- falling back to CEM
2026-08-07 10:27:42,770 | INFO | CEM matching: 8 exact-stratum matches, 1 coarse-backoff matches, 0 random-fallback matches (no comparable stratum at all)
2026-08-07 10:27:42,781 | INFO | Campaign 3: penalized logit unusable (degenerate propensity scores (92% at the extremes)) -- falling back to CEM
2026-08-07 10:27:42,814 | INFO | CEM matching: 5 exact-stratum matches, 1 coarse-backoff matches, 1 random-fallback matches (no comparable stratum at all)
2026-08-07 10:27:42,818 | INFO | Pooling campaign effects (empirical-Bayes shrinkage by campaign type)
2026-08-07 10:27:42,822 | INFO | Pooling complete: 30 campaigns had a direct estimate, 0 were fully pooled from their type group or the grand mean.


,CAMPAIGN,N_TREATED,N_CONTROL,DID_ESTIMATE_PER_WEEK,SHRUNK_ESTIMATE_PER_WEEK,IS_DIRECT_ESTIMATE
16,3,7,1054,0.255614,0.197197,1
21,27,9,1891,-10.010000,0.093018,1


***version control**


In [208]:
# --------------------------------------------------------------------------- #
# Step 7: Doubly Robust Causal Lift Estimation (IPTW + ANCOVA)
# --------------------------------------------------------------------------- #
import numpy as np
import pandas as pd
import logging
import statsmodels.formula.api as smf
from collections import defaultdict

log = logging.getLogger("campaign_pipeline")

def build_doubly_robust_dataset(
    cfg, 
    campaign_desc: pd.DataFrame, 
    tables: dict, 
    pre_days: int = 28
) -> pd.DataFrame:
    """
    Constructs the exact baseline (Pre-Sales) and outcome (Campaign Sales) matrix 
    for all eligible and non-eligible households to enable Doubly Robust estimation.
    """
    log.info("Constructing Pre-Campaign Baseline and Outcome matrix for all households...")
    
    coupon = tables["coupon"]
    coupon_dedup = coupon.drop_duplicates(subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"])
    elig_map = coupon_dedup.groupby("CAMPAIGN")["PRODUCT_ID"].apply(set).to_dict()
    
    campaign_table = tables["campaign_table"]
    
    # Dynamically find the household column
    hh_col = "household_key" if "household_key" in campaign_table.columns else "HOUSEHOLD_KEY"
    if hh_col not in campaign_table.columns:
        hh_col = [c for c in campaign_table.columns if "house" in c.lower()][0]
        
    treated_map = campaign_table.groupby("CAMPAIGN")[hh_col].apply(set).to_dict()
    
    # Define the "Universe" of households (anyone who received ANY campaign)
    universe_hhs = set(campaign_table[hh_col].unique())
    
    # Pre-allocate dictionary: { camp_id: { hh: [PRE_SALES, CAMP_SALES] } }
    data = defaultdict(lambda: defaultdict(lambda: [0.0, 0.0]))
    
    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]].to_dict("index")
    
    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk = chunk[chunk["QUANTITY"] > 0]
        txn_hh_col = "household_key" if "household_key" in chunk.columns else "HOUSEHOLD_KEY"
        
        # Filter chunk to only include our universe of households to save memory
        chunk = chunk[chunk[txn_hh_col].isin(universe_hhs)]
        
        for camp_id, bounds in windows.items():
            pre_start = bounds["START_DAY"] - pre_days
            camp_start = bounds["START_DAY"]
            camp_end = bounds["END_DAY"]
            
            sub = chunk[(chunk["DAY"] >= pre_start) & (chunk["DAY"] <= camp_end)]
            if sub.empty: continue
            
            elig_skus = elig_map.get(camp_id, set())
            sub_elig = sub[sub["PRODUCT_ID"].isin(elig_skus)]
            if sub_elig.empty: continue
            
            # Aggregate Pre-Period Sales (t-28 to t-1)
            pre_df = sub_elig[sub_elig["DAY"] < camp_start]
            for hh, val in pre_df.groupby(txn_hh_col)["SALES_VALUE"].sum().items():
                data[camp_id][hh][0] += val
                
            # Aggregate Campaign Period Sales (t to t_end)
            camp_df = sub_elig[sub_elig["DAY"] >= camp_start]
            for hh, val in camp_df.groupby(txn_hh_col)["SALES_VALUE"].sum().items():
                data[camp_id][hh][1] += val

    # Flatten into a DataFrame
    records = []
    for camp_id, hh_dict in data.items():
        treated_hhs = treated_map.get(camp_id, set())
        
        # We must include all households in the universe to accurately model the controls (including Zeros!)
        for hh in universe_hhs:
            pre_sales = hh_dict.get(hh, [0.0, 0.0])[0]
            camp_sales = hh_dict.get(hh, [0.0, 0.0])[1]
            is_treated = 1 if hh in treated_hhs else 0
            
            records.append({
                "CAMPAIGN": camp_id,
                "household_key": hh,
                "TREATED": is_treated,
                "PRE_SALES": pre_sales,
                "CAMP_SALES": camp_sales
            })
            
    df_dr = pd.DataFrame(records)
    log.info(f"Matrix built successfully. Total shape: {df_dr.shape}")
    return df_dr


def execute_doubly_robust_pipeline(dr_df: pd.DataFrame) -> pd.DataFrame:
    """
    Executes the IPTW + ANCOVA pipeline.
    1. Fits Logistic Regression for Propensity Score e(x).
    2. Calculates Weights W_i = e(x)/(1-e(x)) for controls, trimmed at 99th percentile.
    3. Fits WLS Regression to extract unbiased causal lift.
    """
    log.info("Running Doubly Robust Estimation (IPTW + ANCOVA) across all campaigns...")
    results = []
    
    for camp_id, group in dr_df.groupby("CAMPAIGN"):
        if group["TREATED"].nunique() < 2:
            continue
            
        try:
            # 1. Propensity Score Model
            # Add a tiny constant to PRE_SALES to help convergence if too many exact zeros exist
            group = group.copy()
            group["PRE_SALES_ADJ"] = group["PRE_SALES"] + 0.001 
            
            logit_model = smf.logit("TREATED ~ PRE_SALES_ADJ", data=group).fit(disp=False)
            group["PROPENSITY"] = logit_model.predict(group)
            
            # Bound propensity scores to prevent division by zero or infinite weights
            group["PROPENSITY"] = group["PROPENSITY"].clip(lower=0.001, upper=0.999)
            
            # 2. Assign IPTW Weights
            # Treated = 1.0, Control = e(x) / (1 - e(x))
            group["WEIGHT"] = np.where(
                group["TREATED"] == 1, 
                1.0, 
                group["PROPENSITY"] / (1.0 - group["PROPENSITY"])
            )
            
            # Trim extreme control weights at the 99th percentile to stabilize variance
            weight_cap = group.loc[group["TREATED"] == 0, "WEIGHT"].quantile(0.99)
            group.loc[group["TREATED"] == 0, "WEIGHT"] = group.loc[group["TREATED"] == 0, "WEIGHT"].clip(upper=weight_cap)
            
            # 3. Weighted Least Squares (ANCOVA)
            wls_model = smf.wls("CAMP_SALES ~ TREATED + PRE_SALES", data=group, weights=group["WEIGHT"]).fit()
            
            results.append({
                "CAMPAIGN": camp_id,
                "TRUE_CAUSAL_LIFT": wls_model.params["TREATED"],
                "LIFT_P_VALUE": wls_model.pvalues["TREATED"],
                "SIGNIFICANT_LIFT": bool(wls_model.pvalues["TREATED"] < 0.05),
                "BASELINE_COEF": wls_model.params["PRE_SALES"]
            })
            
        except Exception as e:
            log.warning(f"Campaign {camp_id} failed during Doubly Robust estimation: {e}")
            
    df_results = pd.DataFrame(results)
    log.info(f"Causal estimation complete for {len(df_results)} campaigns.")
    return df_results


# --------------------------------------------------------------------------- #
# Execution Pipeline Call
# --------------------------------------------------------------------------- #
# 1. Build the balanced dataset
dr_dataset = build_doubly_robust_dataset(cfg, tables["campaign_desc"], tables, pre_days=28)

# 2. Estimate True Causal Lift
causal_results = execute_doubly_robust_pipeline(dr_dataset)

# 3. Merge with Margin Economics to view the final causal ROI landscape
final_causal_summary = margin_econ[[
    "CAMPAIGN", "DESCRIPTION", "DISCOUNT_DEPTH_D", "UNPROFITABLE_BY_DESIGN"
]].merge(causal_results, on="CAMPAIGN", how="inner")

# Display the 10 campaigns sorted by their actual true incremental lift
display(final_causal_summary.sort_values("TRUE_CAUSAL_LIFT", ascending=False).head(10))
print(dr_dataset.columns)

2026-08-07 10:12:15,672 | INFO | Constructing Pre-Campaign Baseline and Outcome matrix for all households...
2026-08-07 10:12:17,391 | INFO | Matrix built successfully. Total shape: (47520, 5)
2026-08-07 10:12:17,399 | INFO | Running Doubly Robust Estimation (IPTW + ANCOVA) across all campaigns...
2026-08-07 10:12:17,647 | INFO | Causal estimation complete for 30 campaigns.


,CAMPAIGN,DESCRIPTION,DISCOUNT_DEPTH_D,UNPROFITABLE_BY_DESIGN,TRUE_CAUSAL_LIFT,LIFT_P_VALUE,SIGNIFICANT_LIFT,BASELINE_COEF
7,18,TypeA,1.148230,True,38.213116,2.137112e-10,True,1.503575
12,13,TypeA,1.265135,True,33.249743,6.336850e-11,True,1.139812
1,15,TypeC,0.811757,True,9.941071,1.374206e-17,True,4.504711
0,24,TypeB,1.315216,True,6.599530,7.090993e-10,True,0.869725
8,19,TypeB,1.549335,True,6.547678,1.210439e-22,True,0.632321
17,8,TypeA,1.074211,True,5.791192,1.617919e-01,False,1.139131
2,25,TypeB,1.089864,True,4.744061,1.840544e-18,True,0.514654
4,23,TypeB,1.004728,True,3.982988,2.819824e-11,True,0.824896
27,28,TypeB,0.505999,True,3.957316,7.493930e-19,True,0.528571
26,29,TypeB,0.713900,True,3.572477,1.510311e-13,True,0.986011


Index(['CAMPAIGN', 'household_key', 'TREATED', 'PRE_SALES', 'CAMP_SALES'], dtype='str')


In [209]:
# --------------------------------------------------------------------------- #
# Step 6: Causal Validation & Pre-Treatment Placebo Tests (Patched)
# --------------------------------------------------------------------------- #
import numpy as np
import pandas as pd
import logging
import statsmodels.formula.api as smf

log = logging.getLogger("campaign_pipeline")

def construct_placebo_windows(
    cfg, 
    campaign_desc: pd.DataFrame, 
    tables: dict, 
    placebo_offset_days: int = 28
) -> pd.DataFrame:
    """
    Constructs a placebo outcome matrix by shifting the campaign window backwards 
    by `placebo_offset_days`. Evaluates treated vs. control households.
    """
    log.info(f"Building Placebo outcomes shifted {-placebo_offset_days} days prior to campaigns...")
    
    coupon = tables["coupon"]
    coupon_dedup = coupon.drop_duplicates(subset=["CAMPAIGN", "COUPON_UPC", "PRODUCT_ID"])
    elig_map = coupon_dedup.groupby("CAMPAIGN")["PRODUCT_ID"].apply(set).to_dict()
    
    # Identify Treated Households per Campaign
    campaign_table = tables["campaign_table"]
    
    # --- PATCH: Dynamically find the household column name ---
    if "household_key" in campaign_table.columns:
        hh_col = "household_key"
    elif "HOUSEHOLD_KEY" in campaign_table.columns:
        hh_col = "HOUSEHOLD_KEY"
    else:
        # Fallback to the first column containing 'house'
        matches = [c for c in campaign_table.columns if "house" in c.lower()]
        if not matches:
            raise KeyError(f"Could not locate a household key column in campaign_table. Columns: {campaign_table.columns.tolist()}")
        hh_col = matches[0]
        
    treated_map = campaign_table.groupby("CAMPAIGN")[hh_col].apply(set).to_dict()
    
    # Define Shifted Windows
    windows = campaign_desc.set_index("CAMPAIGN")[["START_DAY", "END_DAY"]].to_dict("index")
    placebo_records = []
    
    for chunk in pd.read_csv(cfg.paths["transaction"], chunksize=cfg.transaction_chunksize):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk = chunk[chunk["QUANTITY"] > 0]
        
        # Dynamically find the household column in the transaction chunk as well, just in case
        txn_hh_col = "household_key" if "household_key" in chunk.columns else "HOUSEHOLD_KEY"
        
        for camp_id, bounds in windows.items():
            # Shift window backwards for placebo test
            p_start = bounds["START_DAY"] - placebo_offset_days
            p_end = bounds["END_DAY"] - placebo_offset_days
            
            # Ensure we don't drop below Day 1
            if p_start < 1: 
                continue 
                
            sub = chunk[(chunk["DAY"] >= p_start) & (chunk["DAY"] <= p_end)]
            if sub.empty:
                continue
                
            elig_skus = elig_map.get(camp_id, set())
            treated_hhs = treated_map.get(camp_id, set())
            
            elig_df = sub[sub["PRODUCT_ID"].isin(elig_skus)]
            
            for hh, g in elig_df.groupby(txn_hh_col):
                is_treated = 1 if hh in treated_hhs else 0
                placebo_records.append({
                    "CAMPAIGN": camp_id,
                    "household_key": hh,
                    "TREATED": is_treated,
                    "PLACEBO_SALES": g["SALES_VALUE"].sum()
                })
                
    df_placebo = pd.DataFrame(placebo_records)
    
    if df_placebo.empty:
        log.warning("Placebo DataFrame is empty! Pre-treatment window might be out of bounds.")
        return df_placebo
        
    # Aggregate to ensure one row per HH-Campaign
    df_placebo = df_placebo.groupby(["CAMPAIGN", "household_key", "TREATED"])["PLACEBO_SALES"].sum().reset_index()
    return df_placebo

def run_placebo_falsification_test(placebo_df: pd.DataFrame) -> pd.DataFrame:
    """
    Runs an OLS regression on the placebo window. 
    The coefficient for 'TREATED' should be statistically indistinguishable from 0.
    """
    log.info("Executing Falsification Check: Estimating DiD on Placebo Windows...")
    results = []
    
    if placebo_df.empty:
        return pd.DataFrame()
        
    for camp_id, group in placebo_df.groupby("CAMPAIGN"):
        # We need variance in treatment assignment to run the model
        if group["TREATED"].nunique() > 1:
            try:
                # Basic OLS for Placebo ATE
                model = smf.ols("PLACEBO_SALES ~ TREATED", data=group).fit()
                p_val = model.pvalues["TREATED"]
                effect = model.params["TREATED"]
                
                results.append({
                    "CAMPAIGN": camp_id,
                    "PLACEBO_EFFECT": effect,
                    "P_VALUE": p_val,
                    "CONFOUNDED_FLAG": True if p_val < 0.05 else False
                })
            except Exception as e:
                pass
                
    results_df = pd.DataFrame(results)
    if not results_df.empty:
        failed = results_df["CONFOUNDED_FLAG"].sum()
        log.info(f"Placebo Test Complete: {failed}/{len(results_df)} campaigns failed the falsification check.")
    
    return results_df

# --------------------------------------------------------------------------- #
# Execution Pipeline Call
# --------------------------------------------------------------------------- #
df_placebo = construct_placebo_windows(cfg, tables["campaign_desc"], tables)
placebo_results = run_placebo_falsification_test(df_placebo)

# Merge results with our Margin Economics table to see the full picture
if not placebo_results.empty:
    validation_summary = margin_econ[["CAMPAIGN", "UNPROFITABLE_BY_DESIGN"]].merge(
        placebo_results, on="CAMPAIGN", how="inner"
    )
    display(validation_summary.sort_values("P_VALUE").head(10))
else:
    log.warning("No placebo results were generated.")

2026-08-07 10:12:17,662 | INFO | Building Placebo outcomes shifted -28 days prior to campaigns...
2026-08-07 10:12:19,746 | INFO | Executing Falsification Check: Estimating DiD on Placebo Windows...
2026-08-07 10:12:19,812 | INFO | Placebo Test Complete: 20/30 campaigns failed the falsification check.


,CAMPAIGN,UNPROFITABLE_BY_DESIGN,PLACEBO_EFFECT,P_VALUE,CONFOUNDED_FLAG
7,18,True,121.308787,2.031199e-77,True
12,13,True,107.413765,5.145982e-76,True
17,8,True,71.899387,9.351676e-59,True
4,23,True,7.466476,1.601840e-17,True
0,24,True,18.495695,6.924002e-14,True
10,14,True,3.313492,9.695581e-14,True
2,25,True,4.714237,3.214605e-12,True
9,17,True,3.428100,2.824692e-07,True
29,26,True,5.763305,3.179910e-07,True
26,29,True,5.004438,1.185183e-05,True


In [210]:
# --------------------------------------------------------------------------- #
# Step 4.6: Translate per-commodity repurchase cycles into a per-campaign
# post-window, then re-run outcome construction and Plan 1 with it.
# --------------------------------------------------------------------------- #
#
# Logic: each campaign's eligible products belong to a set of commodities
# (already computed inside compute_household_campaign_outcomes as
# eligible_commodities_by_campaign). The post-window for that campaign should
# be long enough to catch pull-forward for the SLOWEST-cycling eligible
# commodity, not the fastest -- otherwise pantry-loading on long-cycle goods
# gets cut off and Y_POST_SALES understates the payback effect for exactly
# the products most prone to it. We use the p75 gap (not the median) as the
# per-commodity horizon, per document 3's rule that the horizon must be AT
# LEAST as long as natural purchase frequency -- median would leave half of
# repurchases outside the window by construction.

def build_campaign_post_windows(repurchase_cycles, coupon, product, default_weeks=4, cap_weeks=12):
    """
    Returns {CAMPAIGN: post_period_weeks}, one value per campaign, derived as
    the max p75_gap (in weeks, rounded up) across that campaign's eligible
    commodities. Falls back to default_weeks if a campaign's commodities have
    no repurchase-cycle estimate (too few unpromoted-day observations).
    Capped at cap_weeks to prevent one long-cycle outlier commodity (e.g.
    a durable good bought twice a year) from blowing up the post-window for
    an entire campaign of otherwise fast-cycling products.
    """
    product_commodity = product.set_index("PRODUCT_ID")["COMMODITY_DESC"]
    elig = coupon[["CAMPAIGN", "PRODUCT_ID"]].drop_duplicates()
    elig["COMMODITY_DESC"] = elig["PRODUCT_ID"].map(product_commodity)

    gap_lookup = repurchase_cycles.set_index("COMMODITY_DESC")["p75_gap"]

    campaign_windows = {}
    campaigns_using_default = []
    for campaign, group in elig.groupby("CAMPAIGN"):
        commodities = set(group["COMMODITY_DESC"].dropna())
        gaps = gap_lookup.reindex(list(commodities)).dropna()
        if gaps.empty:
            campaign_windows[campaign] = default_weeks
            campaigns_using_default.append(campaign)
        else:
            weeks = int(np.ceil(gaps.max() / 7.0))
            campaign_windows[campaign] = min(weeks, cap_weeks)

    if campaigns_using_default:
        log.warning(f"{len(campaigns_using_default)} campaigns had no commodity-level repurchase "
                     f"estimate and fell back to the {default_weeks}-week default: {campaigns_using_default}")

    log.info(f"Per-campaign post-windows (weeks): min={min(campaign_windows.values())}, "
             f"median={np.median(list(campaign_windows.values())):.1f}, "
             f"max={max(campaign_windows.values())}")
    return campaign_windows


campaign_post_windows = build_campaign_post_windows(
    repurchase_cycles, dedupe_coupon_table(tables["coupon"]), tables["product"]
)
pd.Series(campaign_post_windows, name="post_period_weeks").rename_axis("CAMPAIGN").to_csv(
    "pipeline_output/campaign_post_windows.csv"
)
campaign_post_windows

2026-08-07 10:12:19,827 | INFO | coupon.csv: dropped 5164 exact-duplicate rows (4.15%)
2026-08-07 10:12:19,859 | INFO | Per-campaign post-windows (weeks): min=5, median=8.0, max=10


{1: 6,
 2: 7,
 3: 8,
 4: 7,
 5: 10,
 6: 5,
 7: 8,
 8: 10,
 9: 8,
 10: 8,
 11: 7,
 12: 7,
 13: 10,
 14: 8,
 15: 5,
 16: 8,
 17: 8,
 18: 10,
 19: 7,
 20: 8,
 21: 8,
 22: 9,
 23: 8,
 24: 7,
 25: 7,
 26: 10,
 27: 8,
 28: 7,
 29: 8,
 30: 8}